# Reproducing Chen, He et al. (2026) from dandiset 001933

Paper: Chen C, He Q, Tombesi G, et al. (2026). Leucine-rich repeat kinase 2 impairs the release sites of
Parkinson's disease vulnerable dopamine axons. Nature Communications.
[doi:10.1038/s41467-026-75194-3](https://doi.org/10.1038/s41467-026-75194-3). This attempt used the open access
full text (CC BY) and its Supplementary Information.

Dandiset: [DANDI:001933](https://dandiarchive.org/dandiset/001933/0.260825.1457), version 0.260825.1457. This version
was asked for: it is the version of the earlier attempt, `001933-chen-2026-01`, and the dandiset has not changed since
(the data check shows the version and the draft hold the same files). It is also the only published version, and the
paper names neither the dandiset nor a version (its data statement points to Zenodo, PRIDE and the BioImage Archive).

This is attempt `001933-chen-2026-02`, under protocol pilot-7. The data are the same as in attempt -01; the protocol
changed. Pilot-7 adds a kind of claim, `data: elsewhere`, for a claim whose data the paper deposited in another public
archive. The eight claims of Figures 1 to 5, blocked in attempt -01 as `data_missing`, are such claims: they now name
their archive and are tested here on those data, counted apart from the dandiset's claims. The ten photometry claims
and their criteria are unchanged and are re-run. The data check also says, for each finding of attempt -01, whether
it still holds.

What the dandiset holds: one session per mouse of GRAB-DA3m fiber photometry in the dorsal striatum during optogenetic
stimulation of SNc dopamine neurons, 15 Anxa1-iCre mice (dorsolateral striatum) and 12 Calb1-Cre mice (dorsomedial
striatum), LRRK2-WT and LRRK2-G2019S (GS). This is the data of Figure 6 and Supplementary Figures 7 and 8.

Outside data: the paper's Zenodo record ([10.5281/zenodo.19930567](https://doi.org/10.5281/zenodo.19930567)): the
source data file, used in `source-data-link` and as the per-section, per-area and per-blot values of Figures 1, 2, 4
and 5, and the Supplementary data files; PRIDE PXD065006 (phosphopeptide abundances) and PXD065125 (pulldown
intensities and the DIA-NN report); and the BioImage Archive, where the synaptosome images of S-BIAD2074 are streamed
and measured and the file lists of S-BIAD2070, 2071 and 2072 are matched to the source data. The paper's photometry
code was read at [DombeckLab/lrrk2_photometry_analysis](https://github.com/DombeckLab/lrrk2_photometry_analysis),
commit d34558f, and its proteomics R code in the Zenodo record (`analysis code.zip`). The Imaris analyses of
Figures 1 and 5 and the Proteome Discoverer search of Figure 2 use commercial software and are not re-run.

| Claim | Paper | Data | Tested |
|---|---|---|---|
| `source-data-link` | Source data of Fig. 6, Supp. Figs 7 and 8 can be recomputed from the dandiset | dandiset with inputs | tested |
| `fig6e-anxa1-evoked-reduced` | Anxa1 evoked release lower in GS at 0.5 mW (p = 1.10e-20) | dandiset | tested |
| `fig6e-anxa1-evoked-per-mouse` | 7 of 8 Anxa1 GS mice reduced; 3743 not | dandiset | tested |
| `supp7c-anxa1-evoked-all-powers` | Reduction at 0.1, 0.5, 1 and 4 mW | dandiset | tested |
| `fig6f-anxa1-spontaneous-reduced` | Anxa1 spontaneous transients smaller in GS (1064 against 798) | dandiset | tested |
| `fig6f-anxa1-spontaneous-per-mouse` | 6 of 8 Anxa1 GS mice reduced; 4651 and 3743 not | dandiset | tested |
| `supp8c-calb1-spontaneous-not-reduced` | Calb1 spontaneous transients not reduced (p = 0.98) | dandiset | tested |
| `supp8e-calb1-evoked-reduced` | Calb1 evoked release reduced at four powers | dandiset | tested |
| `supp8e-calb1-few-mice-reduced` | Only 2 of 7 Calb1 GS mice reduced | dandiset | tested |
| `cohens-d-anxa1-larger` | Cohen's d at 0.5 mW, Anxa1 2.05 against Calb1 0.76 | dandiset | tested |
| `fig1d-lrrk2-enriched-vulnerable` | LRRK2 higher in Aldh1a1+ SNc neurons (p = 0.0008) | elsewhere: S-BIAD2070, Zenodo | tested |
| `fig2b-rab3-phosphorylation` | 289 phosphopeptides change with MLi-2, including RAB3 T86 | elsewhere: PRIDE PXD065006 | tested |
| `fig2eg-prab3-increased` | p-RAB3A and p-RAB3C up in GS, back to WT with MLi-2 | elsewhere: Zenodo | tested |
| `fig3d-rim-binds-unphosphorylated` | RIM1 and RIM2 prefer RAB3A; RILPL1 and RILPL2 prefer p-RAB3A | elsewhere: PRIDE PXD065125 | tested |
| `fig4d-fewer-active-zones` | Fewer bassoon+ TH+/VAMP2+ synaptosomes; larger bassoon | elsewhere: S-BIAD2074, Zenodo | tested |
| `fig4i-rim1-reduced` | RIM1 lower in synaptosomes only | elsewhere: Zenodo | tested |
| `fig5-bassoon-vulnerable-axons` | Bassoon clusters change in Aldh1a1+ and Anxa1+ axons only | elsewhere: S-BIAD2071, Zenodo | tested |
| `fig5j-no-axon-loss` | No loss of TH+ fibres | elsewhere: S-BIAD2072, Zenodo | tested |

Method choices are fixed in the claims file, `claims/001933/10.1038-s41467-026-75194-3.yaml`. In short: the signal is
the dandiset's processed dF/F, the trace the paper's code analyses; the evoked integral follows the Methods (the
light-on period), with the code's baseline; spontaneous transients follow the Methods and `transient_by_threshold.m`.
The claims of Figures 1 to 5 use the legends' tests on the deposited values, and for Figure 4D and 4E also the
Methods' FIJI procedure, re-implemented in `archives.py`, on the images.

In [ ]:
import io
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import archives
import lrrk2

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
sessions = lrrk2.load_all()
by_mouse = {s["mouse"]: s for s in sessions}
print(f"loaded {len(sessions)} files from dandiset {lrrk2.DANDISET_ID} version {lrrk2.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")
# Only the imputation variant of fig3d-rim-binds-unphosphorylated draws random numbers; it has its own
# generator, seeded in that section.

## Data check

Each check of the description against the values is stated under its output. The check covers the dandiset, the findings of attempt -01, and the data of Figures 1 to 5 in other archives.

In [ ]:
info = pd.DataFrame([{
    "mouse": s["mouse"], "subject_id": s["subject_id"], "line": s["line"], "group": s["group"],
    "genotype": s["genotype"], "sex": s["sex"], "date_of_birth": s["date_of_birth"][:10],
    "session_start": s["session_start"][:19],
    "age_months": round((pd.Timestamp(s["session_start"][:10]) - pd.Timestamp(s["date_of_birth"][:10])).days / 30.44, 1),
    "recorded_in": s["location"][s["dff_region"]], "species": s["species"], "strain": s["strain"],
    "dff_md5": s["md5"][:12]} for s in sessions]).sort_values(["line", "group", "mouse"])
print("Files (one session each) by Cre line and genotype:")
print(pd.crosstab(info.line, info.group, margins=True).to_string())
print("\nPaper: Anxa1 8 G2019S and 7 WT mice (Figure 6C); Calb1 7 G2019S and 5 WT mice (Supplementary Figure 8D),"
      "\nafter excluding one Anxa1 G2019S, one Calb1 G2019S and two Calb1 WT mice (Methods).")
paper_anxa1_gs = [4885, 4651, 4652, 3742, 3743, 4004, 4005, 4007]  # mouse IDs in the Figure 6E legend
ours = sorted(info[(info.line == "Anxa1") & (info.group == "GS")].mouse)
print("Anxa1 G2019S mouse IDs in the Figure 6E legend found in the dandiset:", sorted(paper_anxa1_gs) == ours, ours)
print("\nSex by line and genotype:")
print(pd.crosstab([info.line, info.group], info.sex).to_string())
print("\nAge at the session (months), by line: ", info.groupby("line").age_months.agg(["min", "max"]).to_dict("index"))
print("Recording site by line:", info.groupby("line").recorded_in.unique().to_dict())
print("Species and strain:", info.species.unique().tolist(), info.strain.unique().tolist())
print("\nDistinct dF/F traces (md5 of the full series):", info.dff_md5.nunique(), "of", len(info),
      "; distinct session start times:", info.session_start.nunique())
print("\n", info.to_string(index=False))

# The pinned version against the draft: same files and content?
from dandi.dandiapi import DandiAPIClient
client = DandiAPIClient()
def digests(version):
    return {a.path: (a.size, a.get_raw_metadata()["digest"].get("dandi:dandi-etag")) for a in client.get_dandiset(lrrk2.DANDISET_ID, version).get_assets()}
pinned, draft = digests(lrrk2.DANDISET_VERSION), digests("draft")
print(f"\nAssets: pinned version {len(pinned)}, draft {len(draft)}; identical paths, sizes and etags: {pinned == draft}")

The files hold one session each of 15 Anxa1 mice (8 GS, 7 WT) and 12 Calb1 mice (7 GS, 5 WT), the paper's counts after
its exclusions, and the Anxa1 GS mouse IDs are those of the Figure 6E legend. The four excluded mice are not in the
dandiset, which does not say that they were left out. Ages at the session are 6.3 to 7.6 months, consistent with
surgery at 6 months and recording four weeks later. Anxa1 mice are recorded in the dorsolateral and Calb1 mice in the
dorsomedial striatum, as the paper states. No two files share a dF/F trace or a start time, so no recording is
deposited twice. The pinned version and the draft hold the same files.

In [ ]:
rows = []
for s in sessions:
    t, x = s["t"], s["dff"]
    dt = np.diff(t) * 1000
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "samples": len(t),
                 "first_s": round(t[0], 2), "minutes": round((t[-1] - t[0]) / 60, 1),
                 "dt_med_ms": round(np.median(dt), 2), "dt_min_ms": round(dt.min(), 2), "dt_max_ms": round(dt.max(), 2),
                 "dff_min": round(x.min(), 1), "dff_p8": round(np.percentile(x, 8), 2), "dff_max": round(x.max(), 1),
                 "dff405_min": round(s["dff405"].min(), 1), "dff405_max": round(s["dff405"].max(), 1),
                 "max_diff_recomputed": round(np.abs(x - s["dff_recomputed"]).max(), 4),
                 "raw_lag_samples": s["raw_lag"], "raw_r_at_lag0": round(s["raw_r"], 3),
                 "raw_470_samples_per_bin": round(s["raw_n"] / len(t), 2)})
series = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("dF/F series: stated unit", repr(sessions[0]["dff_unit"]), "; description:", sessions[0]["dff_description"].strip())
print(series.to_string(index=False))
print("\nSampling interval over all files (ms): median", series.dt_med_ms.median(), "min", series.dt_min_ms.min(),
      "max", series.dt_max_ms.max())
print("Recording length (min):", series.minutes.min(), "to", series.minutes.max(), "(Methods: 20 min)")
print("Largest difference between the stored dF/F and the code's df_f plus 20-sample moving mean recomputed from the",
      "stored corrected signal (% dF/F):", series.max_diff_recomputed.max())
print("Lag that best aligns the raw 470 nm signal, averaged into each 100 Hz bin, with the corrected signal:",
      sorted(series.raw_lag_samples.unique()), "; correlation at lag 0 from", series.raw_r_at_lag0.min(), "to",
      series.raw_r_at_lag0.max())

The processed series are regular at 10 ms (the shortest interval, 9.5 ms, comes from the 2 kHz grid) and last 20.4 to
25.4 min, a little longer than the Methods' 20 min. The dF/F series is stated in `a.u.`, but its description and its
values are % dF/F: the code's `df_f` and 20-sample moving mean applied to the stored corrected signal give the stored
series to within 0.026 % dF/F. The raw 470 nm signal, averaged into the 100 Hz bins, aligns with the corrected signal
at lag 0 in every file (correlation 0.77 to 0.99), so raw and processed series share a clock. About 8 of the 20 raw
samples per bin are kept for 470 nm, as the demultiplexing describes.

In [ ]:
code_letter = {0.1: "a", 0.25: "b", 0.5: "c", 1.0: "d", 1.5: "e", 2.0: "f", 3.0: "g", 4.0: "h"}
rows = []
for s in sessions:
    p = s["power_in_mW"]
    dur = s["stop_time"] - s["start_time"]
    paper = np.isin(p, lrrk2.PAPER_POWERS)
    gaps = np.diff(s["start_time"])
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "epochs": len(p),
                 "per_power": "/".join(str((p == q).sum()) for q in sorted(set(p))),
                 "powers": len(set(p)), "first_8_codes": "".join(code_letter[q] for q in p[:8]),
                 "paper_powers_in_half": 1 if paper[:32].all() else (2 if paper[32:].all() else 0),
                 "train_s_paper": round(np.median(dur[paper]), 3), "pulse_ms_paper": s["pulse_length_in_ms"][paper][0],
                 "period_ms_paper": s["period_in_ms"][paper][0],
                 "train_s_other": round(np.median(dur[~paper]), 3), "pulse_ms_other": s["pulse_length_in_ms"][~paper][0],
                 "period_ms_other": s["period_in_ms"][~paper][0],
                 "min_gap_s": round(gaps.min(), 1), "median_gap_s": round(np.median(gaps), 1),
                 "wavelength": s["wavelength_in_nm"][0], "first_onset_s": round(s["start_time"][0], 1),
                 "onset_minus_series_start_s": round(s["start_time"][0] - s["t"][0], 1)})
opto = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Epochs table description (first file):", sessions[0]["opto_description"])
print("\nPowers in the files (mW):", sorted(set(np.concatenate([s["power_in_mW"] for s in sessions]))))
print(opto.to_string(index=False))
print("\nTrain types carrying the paper's four powers, by line and genotype:")
print(opto.groupby(["line", "group", "train_s_paper", "pulse_ms_paper", "period_ms_paper"]).size().to_string())
odd = [(s["mouse"], j) for s in sessions for j in np.where((s["pulse_length_in_ms"] == 0) | np.isnan(s["period_in_ms"]))[0]]
print("\nEpochs with a pulse length of 0 or no period (mouse, epoch, power, length in s, pulses):",
      [(m, int(j), by_mouse[m]["power_in_mW"][j], round(by_mouse[m]["stop_time"][j] - by_mouse[m]["start_time"][j], 2),
        int(by_mouse[m]["number_pulses_per_pulse_train"][j])) for m, j in odd])
print("Shortest gap between onsets in any file (s):", opto.min_gap_s.min(), "(Methods: at least 20 s)")
gap_all = np.concatenate([np.diff(s["start_time"]) for s in sessions])
print("Gaps between onsets over all files (s): 5th, 50th, 95th percentiles", np.percentile(gap_all, [5, 50, 95]).round(1),
      "; fraction under 20 s:", round(float((gap_all < 20).mean()), 3))

Every file holds 64 trains, 8 at each of 8 powers from 0.1 to 4 mW; the paper uses four of them (0.1, 0.5, 1 and 4 mW).
The table's own description gives 8 ms on and 8 ms off for 500 ms and only those four powers, but the table holds two
kinds of train. In Anxa1 files one half of the session has 9 ms pulses every 10 ms for 0.33 s, the other 8 ms pulses
every 16 ms for 0.51 s; Calb1 files hold only the second kind. The paper's four powers fall on 0.51 s trains in 25
files and on 0.33 s trains in two Anxa1 GS mice (3742 and 4885), so these two mice were stimulated differently from
every WT mouse; a variant of `fig6e-anxa1-evoked-reduced` leaves them out. One train each in mice 4652 and 740 has a
pulse length of 0, no period and one pulse, though it lasts 0.51 s like its neighbours, so its pulse columns are
wrong. Trains of the same power come in pairs 15.3 to 15.5 s apart, and 51% of the gaps are under 20 s, where the
Methods and the table description say at least 20 s.

In [ ]:
rows = []
for s in sessions:
    shifted = dict(s, onset=s["onset"] + 1000, offset=s["offset"] + 1000)  # 10 s later, between trains
    true = lrrk2.evoked_integrals(s, "fixed")
    moved = lrrk2.evoked_integrals(shifted, "fixed")
    iso = lrrk2.evoked_integrals(s, "fixed", trace="dff405")
    traces = np.array([s["dff"][i - 100:i + 300] for i in s["onset"] if i + 300 <= len(s["dff"])])
    mean_trace = traces.mean(0)
    ok = ~np.isnan(true)
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"],
                 "median_at_onsets": np.nanmedian(true), "median_10s_later": np.nanmedian(moved),
                 "median_405_at_onsets": np.nanmedian(iso),
                 "peak_latency_ms": (np.argmax(mean_trace) - 100) * 10,
                 "rise_before_onset": mean_trace[90:100].mean() - mean_trace[:90].mean(),
                 "spearman_power": stats.spearmanr(s["power_in_mW"][ok], true[ok])[0],
                 **{f"{q}mW": np.nanmedian(true[s["power_in_mW"] == q]) for q in sorted(set(s["power_in_mW"]))}})
clock = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Integral from 0 to 0.5 s after onset minus the mean of -1.0 to -0.1 s (% dF/F x s), median over each file's 64",
      "trains, at the table's onsets, 10 s later, and for the 405 nm control; latency of the peak of the mean trace;",
      "Spearman correlation of the integral with the labelled power; median integral per labelled power.")
print(clock.round(2).to_string(index=False))
print("\nFiles where the median at onsets exceeds the median 10 s later:", int((clock.median_at_onsets > clock.median_10s_later).sum()),
      "of", len(clock), "; Wilcoxon p =", f"{stats.wilcoxon(clock.median_at_onsets, clock.median_10s_later).pvalue:.2g}")
print("Peak latency (ms) from", clock.peak_latency_ms.min(), "to", clock.peak_latency_ms.max(),
      "; mean rise in the 0.1 s before onset from", clock.rise_before_onset.min().round(2), "to", clock.rise_before_onset.max().round(2))
print("Spearman correlation with power, median by line:", clock.groupby("line").spearman_power.median().round(2).to_dict())

# The two Anxa1 G2019S mice whose paper powers are 0.33 s trains, against two GS mice given 0.51 s trains,
# one epoch per row: labelled power, train type and the 0 to 0.5 s integral.
rows = []
for m in (3742, 4885, 4004, 4651):
    s = by_mouse[m]
    v = lrrk2.evoked_integrals(s, "fixed")
    for j in range(len(v)):
        rows.append({"mouse": m, "epoch": j, "power": s["power_in_mW"][j],
                     "train_s": round(s["stop_time"][j] - s["start_time"][j], 2),
                     "train": "0.33 s, 9/10 ms" if s["period_in_ms"][j] == 10 else "0.51 s, 8/16 ms", "pulse_ms": s["pulse_length_in_ms"][j],
                     "period_ms": s["period_in_ms"][j], "integral_0_0.5s": v[j]})
ep = pd.DataFrame(rows)
print("\nEpoch by epoch, mice 3742 and 4885 (0.33 s trains at the paper's powers) and 4004 and 4651 (0.51 s trains):")
for m in (3742, 4885):
    print(ep[ep.mouse == m].round(2).to_string(index=False))
print("\nMedian 0 to 0.5 s integral by labelled power and train type:")
print(ep.pivot_table(index=["mouse", "train"], columns="power", values="integral_0_0.5s", aggfunc="median").round(1).to_string())
print("Spearman correlation of the integral with labelled power, within each train type:")
print(ep.groupby(["mouse", "train"]).apply(lambda g: round(stats.spearmanr(g.power, g["integral_0_0.5s"], nan_policy="omit")[0], 2)).to_string())

# Anxa1 files where 1.5, 2 and 3 mW are 0.33 s trains and 0.5 and 1 mW are 0.51 s trains: do the former fall below?
rows = []
for s in sessions:
    if s["line"] != "Anxa1":
        continue
    v = lrrk2.evoked_integrals(s, "fixed")
    kind = {q: set(s["period_in_ms"][s["power_in_mW"] == q]) for q in (0.5, 1.0, 1.5, 2.0, 3.0)}
    if not (all(kind[q] == {10} for q in (1.5, 2.0, 3.0)) and all(kind[q] == {16} for q in (0.5, 1.0))):
        continue
    med = {q: np.nanmedian(v[s["power_in_mW"] == q]) for q in kind}
    rows.append({"mouse": s["mouse"], "group": s["group"], **{f"{q}mW": med[q] for q in kind},
                 "high_033_below_low_051": max(med[q] for q in (1.5, 2.0, 3.0)) < min(med[q] for q in (0.5, 1.0))})
split = pd.DataFrame(rows)
print("\nAnxa1 files with 1.5, 2 and 3 mW as 0.33 s trains and 0.5 and 1 mW as 0.51 s trains, median 0 to 0.5 s integral:")
print(split.round(1).to_string(index=False))
print("Files where every 0.33 s median (1.5, 2, 3 mW) is below both 0.51 s medians (0.5, 1 mW):",
      int(split.high_033_below_low_051.sum()), "of", len(split))


At the table's onsets the 0 to 0.5 s integral has a median of 13 to 260 % dF/F x s per file; 10 s later, between
trains, it is near zero in every file (lower in 27 of 27, Wilcoxon p = 1.5e-08), and the 405 nm control shows no
response (medians -0.74 to 1.15). The mean response peaks 160 to 370 ms after onset. The mean trace rises by up to
25 % dF/F in the 0.1 s before onset: the 20-sample moving mean of the stored dF/F spreads the response about 0.1 s
earlier, which is why the code's baseline ends 0.1 s before onset. The epochs table and the photometry share a clock.

The response grows with the labelled power in every Calb1 file (Spearman 0.76 to 0.94) but not in many Anxa1 files
(median 0.06). Of the 12 Anxa1 files where 1.5, 2 and 3 mW are given as 0.33 s trains and 0.5 and 1 mW as 0.51 s
trains, the 0.33 s responses fall below both 0.51 s ones in 7 (for example mouse 4401) and not in the other 5 (in 4398,
173.5 at 1.5 mW against 114.8 at 0.5 mW). The power labels are consistent with the data in Calb1
files; in Anxa1 files the data alone cannot confirm them. The labels of the four powers the paper uses are confirmed
against its source data under `source-data-link`.

The epoch-by-epoch tables show mice 3742 and 4885, whose paper powers are all 0.33 s trains of 9 ms pulses every
10 ms, and two GS mice given 0.51 s trains. In 3742 the response does not follow the labelled power: within the 0.33 s
trains the median 0 to 0.5 s integral is lowest at 0.5 mW (5.7 % dF/F x s) and higher at 0.1 mW (33.5), and the
correlation with power is negative within each train type. In 4885 the 0.33 s trains give medians of 35.0 to
53.5 at the four labelled powers, below every 0.51 s power (56.1 to 78.1), and the correlation with power is negative
within both train types (-0.28 and -0.74). In 4004 and 4651 the response rises with power within the 0.51 s trains
(Spearman 0.81 and 0.86) and not within the 0.33 s ones (-0.41 and -0.21). The data cannot confirm the power labels of 3742 and 4885.

### Findings of attempt 001933-chen-2026-01

The same version is analysed again, so each finding of the earlier attempt is checked against the files, the archive's
metadata, the paper's code repository and the paper's Crossref record.

In [ ]:
# Each finding of attempt 001933-chen-2026-01, checked again on version 0.260825.1457.
import json
import subprocess

import requests

meta = client.get_dandiset(lrrk2.DANDISET_ID, lrrk2.DANDISET_VERSION).get_raw_metadata()
md5_src, src = lrrk2.load_source_data()


def nearest_epoch(record):
    s = by_mouse[record["mouse"]]
    return min(np.nanmax(np.abs(lrrk2.code_chunk(s, j) - record["trace"])) for j in range(len(s["onset"])))


near = {k: np.array([nearest_epoch(d) for d in src[k]]) for k in ("6C", "S8D")}
print("F1 (Calb1 dF/F is not the paper's). Published triggered traces within 1e-6 of some epoch of the same mouse, any power:",
      {k: f"{int((v < 1e-6).sum())} of {len(v)}" for k, v in near.items()}, f"; closest S8D trace {near['S8D'].min():.3g} % dF/F")
summary = meta["assetsSummary"]
related = meta.get("relatedResource", [])
text = ((meta.get("description") or "") + " " + " ".join(r.get("url", "") for r in related)).lower()
print("F2 (only the photometry is in the dandiset):", summary["numberOfFiles"], "NWB files; the description or related resources",
      "name Zenodo, PRIDE or the BioImage Archive:", any(w in text for w in ("zenodo", "pride", "biostudies", "bioimage", "s-biad", "pxd")))
print("   related resources:", [(r.get("relation"), r.get("url")) for r in related])
m = by_mouse[4398]
print(f"F3 (series of 4398 cut short): last onset {m['start_time'][-1]:.2f} s, series ends {m['t'][-1]:.2f} s,",
      f"{m['t'][-1] - m['start_time'][-1]:.2f} s after it")
print("F4 (epochs table description):", " ".join(sessions[0]["opto_description"].split())[:260])
print("   powers in the tables:", sorted(set(np.concatenate([s["power_in_mW"] for s in sessions]))),
      "; train lengths (s):", sorted({round(float(d), 2) for s in sessions for d in s["stop_time"] - s["start_time"]}),
      "; trains with a pulse length of 0:", sum(int((s["pulse_length_in_ms"] == 0).sum()) for s in sessions))
paper_train = {s["mouse"]: float(np.median((s["stop_time"] - s["start_time"])[np.isin(s["power_in_mW"], lrrk2.PAPER_POWERS)]))
               for s in sessions}
print("F5 (mice given the paper's powers as 0.33 s trains):", {k: round(v, 2) for k, v in paper_train.items() if v < 0.4})
print("F6 (dF/F unit):", sorted({s["dff_unit"] for s in sessions}))
print("F7 (archive summary): approach", [a["name"] for a in summary.get("approach", [])], "; techniques",
      [a["name"] for a in summary.get("measurementTechnique", [])], "; variables", summary.get("variableMeasured"))
print("F8 (exclusions undocumented): the description mentions an exclusion:", "exclu" in (meta.get("description") or "").lower())
head = subprocess.run(["git", "ls-remote", "https://github.com/DombeckLab/lrrk2_photometry_analysis", "HEAD"],
                      capture_output=True, text=True, timeout=120, check=True).stdout.split()[0]
print("F9 to F11 (published code): HEAD of DombeckLab/lrrk2_photometry_analysis is", head[:7], "; attempt -01 read d34558f")
cr = requests.get("https://api.crossref.org/works/10.1038/s41467-026-75194-3", timeout=60)
cr.raise_for_status()
cm = cr.json()["message"]
print("F12 and F13 (paper): Crossref lists a correction or update:", bool(cm.get("updated-by") or cm.get("update-to")),
      "; references naming DANDI:", sum("dandi" in json.dumps(r).lower() for r in cm.get("reference", [])), "of",
      len(cm.get("reference", [])))

None of the findings of attempt -01 is fixed; the version is the same, and so are the code and the paper.

| Finding of attempt -01 | Kind | Now | Shown by |
|---|---|---|---|
| The Calb1 dF/F is not the trace behind Supplementary Figure 8 | data_missing | still present | F1: 0 of 384 Supplementary 8D traces within 1e-6 of any epoch of the mouse (closest 15.7 % dF/F), against 120 of 120 for Figure 6C |
| Only the photometry is in the dandiset; Figures 1 to 5 are elsewhere | data_missing | still present, kind changed | F2: 27 files, and neither the description nor the related resources name Zenodo, PRIDE or the BioImage Archive. Under pilot-7 these data count as public elsewhere, so the finding is recorded again as `other` |
| The series of 4398 ends 0.8 s after its last onset | other | still present | F3: 0.81 s |
| The epochs table description gives one train and four powers | metadata_error | still present | F4: eight powers, trains of 0.32 to 0.52 s, two trains with a pulse length of 0 |
| Two Anxa1 GS mice received 0.33 s trains | other | still present | F5: 3742 and 4885 |
| The dF/F unit is `a.u.` | metadata_error | still present | F6 |
| The archive summary lists a behavioral approach only | metadata_error | still present | F7 |
| The four excluded mice are left out without a note | selection_undocumented | still present | F8 |
| Three disagreements of the published code with the paper | code_mismatch | still present | F9 to F11: the repository is at the commit read in attempt -01 |
| Welch, one-tailed and two-tailed tests named for the per-mouse comparisons | paper_inconsistency | still present | F12: no correction is registered |
| The paper does not cite the dandiset | other | still present | F13: 0 of 99 references name DANDI |

### The data in other archives

The claims of Figures 1 to 5 are tested on the data the paper deposited elsewhere (protocol pilot-7, `data:
elsewhere`): the tabular values behind each figure on Zenodo, the search outputs in PRIDE and the images in the
BioImage Archive. Each is read in memory. This check lists what each archive holds, counts it against the legends,
and looks for values or files deposited twice.

In [ ]:
rec = requests.get(archives.ZENODO_RECORD, timeout=60).json()
print(f"Zenodo record {rec['id']}, created {rec['created'][:10]}, modified {rec['modified'][:10]}:")
for f in rec["files"]:
    print(f"   {f['key']}: {f['size']} bytes, {f['checksum']}")
md5_wb, _ = archives.source_workbook()
print("Source data.xlsx read here, md5", md5_wb, "; equal to the record's:", md5_wb == archives.SOURCE_DATA_MD5)

f1 = archives.fig1d()
f2 = archives.fig2eg()
f4de = archives.fig4de()
blots = archives.fig4()
f5 = archives.fig5de()
f5gh = archives.fig5gh()
f5j = archives.fig5j()
print("\nFigure 1D: sections per mouse", f1.groupby("mouse").size().to_dict(), "; total", len(f1), "(legend: 16 sections, 3 mice)")
print("   LRRK2 intensity, Aldh1a1- from", f1.neg.min().round(3), "to", f1.neg.max().round(3), "; Aldh1a1+ from", f1.pos.min().round(3),
      "to", f1.pos.max().round(3), "(normalised; no unit given)")
print("Figure 2E and 2G: values per group", {p: {g: len(v) for g, v in d.items()} for p, d in f2.items()}, "(legend: 10 and 7 mice)")
print("   ranges:", {p: {g: (round(v.min(), 3), round(v.max(), 3)) for g, v in d.items()} for p, d in f2.items()})
print("Figure 4D and 4E: areas per replicate (mouse) and group (legend: 4D 39 WT and 35 GS areas, 3 mice each;",
      "4E 43 WT and 57 GS areas, 4 mice each):")
print(pd.crosstab([f4de.panel, f4de.replicate], f4de.group, margins=True).to_string())
print("   ranges:", f4de.groupby(["panel", "group"]).value.agg(["min", "max"]).round(4).to_dict("index"))
print("   4D values under 1 %:", f4de[(f4de.panel == "4D") & (f4de.value < 1)][["replicate", "area", "group", "value"]].to_dict("records"))
print("Figure 4G to 4L: values per genotype", {k: (len(v["WT"]), len(v["GS"])) for k, v in blots.items()}, "(legend: 6 mice per genotype)")
print("Figure 5D and 5E: sections with a value, by cell type and group (legend 5D: WT+ 19, GS+ 21, WT- 18, GS- 17;",
      "5E: 19, 20, 16, 17):")
print(f5.dropna().groupby(["panel", "cell", "group"]).size().unstack().to_string())
print("   mice per group:", f5.groupby("group").mouse.unique().apply(list).to_dict())
print("   ranges:", f5.groupby(["panel", "cell"]).value.agg(["min", "max"]).round(4).to_dict("index"))
print("Figure 5G and 5H: sections per group", f5gh.groupby(["panel", "group"]).size().to_dict(), "(legend: 16 WT, 3 mice; 18 GS, 4 mice);",
      "the animal column lists", sorted(f5gh.animal_column.unique()), "for both groups")
print("Figure 5J: sections per group", f5j.groupby("group").size().to_dict(), "(legend: 12 sections, 4 mice each); the animal column",
      "lists", sorted(f5j.animal_column.unique()), "for both groups")

# Values deposited twice: whole sections of Figure 5D/5E with the same four values, and repeated values within any panel.
wide = f5.set_index(["group", "column", "mouse", "section", "panel", "cell"]).value.unstack(["panel", "cell"])
dups = wide[wide.round(9).duplicated(keep=False)]
print("\nFigure 5D/5E sections whose four values (density and volume, Aldh1a1+ and -) equal another section's:")
print(dups.to_string())


def repeats(v):
    v = pd.Series(v).dropna()
    return sorted(v[v.duplicated(keep=False)].round(9).unique().tolist())


rep = {"1D": repeats(np.r_[f1.pos, f1.neg]), **{f"2/{p}": repeats(np.concatenate(list(d.values()))) for p, d in f2.items()},
       **{f"{p}": repeats(g.value) for p, g in f4de.groupby("panel")}, **{k: repeats(np.r_[v["WT"], v["GS"]]) for k, v in blots.items()},
       **{f"{p}": repeats(g.value) for p, g in f5gh.groupby("panel")}, "5J": repeats(f5j.value),
       **{f"{p} {c}": repeats(g.value) for (p, c), g in f5.groupby(["panel", "cell"])}}
print("Repeated values within each panel:", {k: v for k, v in rep.items() if v})

The source data file read here is the one in the Zenodo record (equal md5). Its counts agree with the legends for
Figures 1D (16 sections, 3 mice), 2E and 2G (10 and 7 per group), 4D (39 WT and 35 GS areas), 4G to 4L (6 mice per
genotype), 5D, 5G, 5H and 5J. They disagree in two places: Figure 4E has 51 GS areas where the legend gives 57, and
Figure 5E has 21 GS Aldh1a1+ sections where the legend gives 20. One GS value of Figure 4D, 0.0766 %, is of the size
of the Figure 4E areas (0.04 to 0.15) and may have been pasted from there; it is kept, and leaving it out is shown.
Sections 4 and 5 of WT mouse 1723 in Figure 5D and 5E carry the same four values to six digits, so one section
appears to have been entered twice; as the claims file fixes, it is counted once, and the values as deposited are a
variant. Figure 5D holds a density of 0 for one WT Aldh1a1- section, kept as deposited. The other repeated values
(one in 4D, one in 5E) are single values, not whole sections. The sheets of Figure 5G/5H and 5J give one animal
column for both genotypes, holding WT IDs (1021, 1024, 1028; 3244, 3757, 3874, 4434), so the GS animals of these
panels are not identified in the source data. The values carry no units; Figure 1D and 5J are normalised to about 1.

In [ ]:
supp, md5_supp = archives.supplementary_data()
print("Supplementary files.zip md5", md5_supp, "; files:", sorted(supp))
pp, md5_pp = archives.phosphopeptides()
print("\nPRIDE PXD065006 peptide groups table: md5", md5_pp, ";", len(pp), "peptides,", int(pp.Modifications.str.contains("Phospho").sum()),
      "with a phosphorylation")
ab = [c for c in pp.columns if c.startswith("Abundance")]
print("   abundance columns:", ab)
print("   values per sample:", pp[ab].notna().sum().to_dict())
print("   peptides with all six values", int(pp[ab].notna().all(axis=1).sum()), "; with at least two per group",
      int(((pp[ab[:3]].notna().sum(axis=1) >= 2) & (pp[ab[3:]].notna().sum(axis=1) >= 2)).sum()))
s1 = pd.read_excel(io.BytesIO(supp["Supplementary file 1.xlsx"]), sheet_name=None)
same = s1["PeptideGroups"][ab].fillna(-1).to_numpy().tolist() == pp[ab].fillna(-1).to_numpy().tolist()
print("   equal to the PeptideGroups sheet of Supplementary data 1:", same, "; that file's sheets:",
      {k: v.shape for k, v in s1.items()})
rab3 = pp[pp["Annotated Sequence"].str.contains("YRTITTAYYR", na=False)]
print("   RAB3 peptide covering T86:", rab3[["Annotated Sequence", "Modifications", "Modifications in Master Proteins"]].to_dict("records"))

pull, md5_pull, pull_cols = archives.pulldown()
print("\nPRIDE PXD065125 FilteredDataset.pg_matrix.Brain_Rab3A.csv: md5", md5_pull, ";", len(pull), "protein groups; columns",
      pull_cols[3:])
report = pd.read_csv(io.BytesIO(archives.fetch(archives.PULLDOWN_REPORT_URL)[0]), sep="\t")
runs = [c for c in report.columns if c.endswith(".raw")]
raw = pd.read_csv(io.BytesIO(archives.fetch(archives.PULLDOWN_URL)[0]))
order = {}
for c in [c for c in raw.columns if c.startswith(("phosRab3Brain", "Rab3Brain"))]:
    v = raw.set_index("Protein.Group")[c]
    for r in runs:
        w = report.set_index("Protein.Group").reindex(v.index)[r]
        if np.allclose(v.fillna(-1), w.fillna(-1)):
            order[c] = r.split("\\")[-1]
print("   each pulldown column equals the run of report.pg_matrix.tsv:", order)
sdrf = pd.read_csv(io.BytesIO(archives.fetch(archives.PULLDOWN_SDRF_URL)[0]), sep="\t")
print("   SDRF bait per run:", sorted((n.split("_")[3], n.split("_")[4]) for n in sdrf["source name"] if n.startswith("20240406")))
print("   the six RAB3A and six p-RAB3A runs are paired here in run order, C13 with C19 and so on")

The peptide groups table of PXD065006 equals the PeptideGroups sheet of Supplementary data 1: 8008
phosphopeptides with three vehicle and three MLi-2 samples, of which 5366 have all six values and 6070 at least two
per group. It holds the RAB3 peptide YRTITTAYYR phosphorylated at T86 of RAB3A and RAB3B (T92 of RAB3C). The
paper's volcano table, also in Supplementary data 1, holds 6157 peptides. The pulldown matrix of PXD065125 holds
3480 protein groups, the paper's count. Its twelve pulldown columns equal runs C13 to C24 of the DIA-NN report, and
the SDRF gives p-RAB3A for C13 to C18 and RAB3A for C19 to C24, so the column labels agree with the runs. The
pairing of replicates behind the paired test of Figure 3E and 3F is not stated; run order is used, as the claims file
fixes.

In [ ]:
bia = {acc: archives.bia_files(acc) for acc in archives.BIA_FILE_LISTS}
for acc, d in bia.items():
    print(f"{acc}: {len(d)} files, {d['size'].sum() / 1e9:.2f} GB; folders", d.path.str.split("/").str[-2].value_counts().to_dict())
    print("   channel annotations:", d.filter(like="Channel").drop_duplicates().to_dict("records"))

import re


def ids(paths, pattern):
    out = []
    for p in paths:
        m = re.search(pattern, p.split("/")[-1])
        out.append(m.groups() if m else (None, None))
    return out


d = bia["S-BIAD2070"]
mice = pd.Series([int(m) for _, m in ids(d.path, r"(C57)-(\d+)-")])
print("\nS-BIAD2070 (Figure 1): images per mouse", mice.value_counts().sort_index().to_dict(),
      "; source data sections per mouse", f1.groupby("mouse").size().to_dict())
d = bia["S-BIAD2071"]
folder = d.path.str.split("/").str[-2]
g = pd.DataFrame(ids(d.path, r"(?:Aldh|Anxa)(GS|WT)-(?:R-)?(?:Middle-)?(\d{3,})?"), columns=["group", "mouse"]).assign(folder=folder.values)
print("S-BIAD2071 (Figure 5): images per folder, group and mouse:")
print(g.fillna("none").groupby(["folder", "group", "mouse"]).size().to_string())
print("   source data mice, 5D/5E:", f5.groupby("group").mouse.unique().apply(list).to_dict(), "; 5G/5H animal column:",
      sorted(f5gh.animal_column.unique()))
d = bia["S-BIAD2072"]
d = d[d.path.str.contains("TH-6M-volume")]
g = pd.DataFrame(ids(d.path, r"(C57|GSHom)-(\d+)-"), columns=["group", "mouse"])
print("S-BIAD2072 (Figure 5J, folder TH-6M-volume): images per group and mouse", g.groupby(["group", "mouse"]).size().to_dict(),
      "; source data animal column", sorted(f5j.animal_column.unique()), "; legend: 8 images per section, 12 sections, 4 mice per group")
d = bia["S-BIAD2074"]
g = pd.DataFrame(ids(d.path, r"Export_(wt|ki)(\d+)_"), columns=["group", "mouse"]).assign(folder=d.path.str.split("/").str[-2].values)
g["group"] = g.group.str.lower()
print("S-BIAD2074 (Figure 4D, 4E): images per folder, group and mouse:")
print(g.groupby(["folder", "group", "mouse"]).size().unstack(fill_value=0).to_string())

The images match the source data by mouse only in part, and none of the studies records which images the figures used.
S-BIAD2070 holds 7 images for each of the three Figure 1D mice, where the source data give 5 or 6 sections each.
S-BIAD2071 holds Aldh1a1 images of the four GS mice of Figure 5D (1272, 1522, 517, 796) and of WT mice 1923, 1929, 516
and 797, where the source data name 1723 for the first; two WT files carry no mouse ID. Its Anxa1 folder holds four
WT mice (1018, 1021, 1024, 1028), where the legend and the source data have three (1018 is not in the source data),
and four GS mice (1019, 1020, 1022, 1052) that the source data do not name. In S-BIAD2072, the volume folder holds 16
images for each of the four WT mice of Figure 5J and of four GS mice (3243, 3246, 4120, 4369), 64 per genotype, where
the legend gives 8 images per section for 12 sections (96). S-BIAD2074 names its images by genotype and a mouse
number from 1 to 5. Its counts do not follow the source data image by image: in the BSN folder the GS mice have 14,
10 and 12 images against 14, 11 and 10 areas in Figure 4D (WT 14, 10, 15 against 14, 11, 14); in the BSN-area folder
GS has 49 images against 51 areas (and the legend's 57) and WT 43 against 43. Part B of `fig4d-fewer-active-zones`
therefore compares genotypes over the deposited images and cannot match them to the published areas.

In [ ]:
# Every synaptosome image of S-BIAD2074, streamed and measured with the Methods' FIJI procedure (archives.py).
t0 = time.time()
syn = archives.analyse_synaptosomes(bia["S-BIAD2074"], workers=12)
print(f"{len(syn)} images streamed and measured in {time.time() - t0:.0f} s; shapes {syn['shape'].value_counts().to_dict()};",
      f"pixel sizes (um) {sorted(syn.pixel_um.unique())}")
dupe = syn[syn.md5.duplicated(keep=False)].sort_values("md5")
within = dupe.groupby(["md5"]).folder.agg(lambda f: f.duplicated().any())
print("Images with the same content as another:", len(dupe), "; pairs within one folder:", int(within.astype(bool).sum()))
print(dupe.groupby("md5").path.apply(lambda p: " = ".join(x.split("/", 1)[1] for x in p)).to_string(index=False))
frac = syn[syn.folder == "BSN"]
area = syn[syn.folder == "BSN-area"]
print("\nBSN folder: VAMP2 particles per image", int(frac.vamp2_particles.min()), "to", int(frac.vamp2_particles.max()),
      "(median", int(frac.vamp2_particles.median()), "); TH+ VAMP2+ per image", int(frac.th_vamp2.min()), "to", int(frac.th_vamp2.max()),
      "; images with none:", int((frac.th_vamp2 == 0).sum()), "(the legend: about 1000 synaptosomes per area)")
print("BSN-area folder: bassoon and TH overlaps per image", int(area.overlaps.min()), "to", int(area.overlaps.max()),
      "; VAMP2+ overlaps", int(area.vamp2_pos.min()), "to", int(area.vamp2_pos.max()), "; images with none:", int((area.vamp2_pos == 0).sum()))
print("\nExcitation wavelengths (nm) of channels 0, 1 and 2 in each image's OME metadata:", syn.excitation_nm.value_counts().to_dict(),
      "; the procedure reads them as bassoon (488), TH (568) and VAMP2 (647)")
print("Images whose metadata give another order:", syn[syn.excitation_nm != "488/561/640"].path.tolist())
print("\nBSN folder: objects above the Default threshold before the size and circularity limits, VAMP2 objects within the",
      "size limits alone, and VAMP2 particles after both, per image (the legend: about 1000 synaptosomes per area):")
print(frac[["bassoon_objects", "th_objects", "vamp2_objects", "vamp2_size_only", "vamp2_particles", "th_vamp2", "bassoon_th_vamp2"]]
      .describe().loc[["min", "25%", "50%", "75%", "max"]].round(0).to_string())
cols = ["group", "mouse", "image", "bassoon_objects", "th_objects", "vamp2_objects", "vamp2_size_only", "vamp2_particles", "th_vamp2", "bassoon_th_vamp2", "percent"]
print("Three images per genotype:")
print(frac.sort_values(["group", "mouse", "image"]).groupby("group").head(3)[cols].to_string(index=False, float_format=lambda v: f"{v:.1f}"))

# One image at the median VAMP2 particle count, with the VAMP2 segmentation drawn on it.
from skimage.segmentation import find_boundaries

row = frac.iloc[(frac.vamp2_particles - frac.vamp2_particles.median()).abs().argsort().iloc[0]]
img, _, px, _ = archives.read_synaptosome_image(row.path)
vamp = archives.rolling_ball(img[2])
thr = archives.ij_default_threshold(vamp)
kept = archives.particles(vamp, px, (0.04, 0.40))
fig, ax = plt.subplots(1, 2, figsize=(12, 6))
hi = np.percentile(vamp, 99.5)
for a, title in zip(ax, ("VAMP2 (647 nm) after background subtraction", "the same, with the segmentation")):
    a.imshow(np.clip(vamp, 0, hi), cmap="gray", vmin=0, vmax=hi)
    a.set_title(title, fontsize=9)
    a.axis("off")
ax[1].contour(vamp >= thr, levels=[0.5], colors="tab:orange", linewidths=0.4)
ax[1].contour(find_boundaries(kept), levels=[0.5], colors="tab:cyan", linewidths=0.6)
fig.suptitle(f"{row.path.split('/')[-1]}: orange, above the Default threshold; cyan, the {int(row.vamp2_particles)} particles kept", fontsize=9)
plt.show()
print(f"Shown: {row.path}; intensities clipped at the 99.5th percentile, {hi:.0f}; {100 * (vamp > hi).mean():.2f} % of pixels above it;",
      f"Default threshold {thr:.0f}; {100 * (vamp >= thr).mean():.2f} % of pixels above the threshold")


All 168 synaptosome images were streamed and measured, and no two share content, although the BSN and BSN-area
folders share file names for mice 1 and 3. One image (ki4 BSN1) is 512 x 512 at 0.0615 um per pixel, the others
1024 x 1024 at 0.0308 or 0.0311 um; particle sizes use each image's own pixel size. With the Methods' procedure as
re-implemented here, an image holds 1 to 417 VAMP2 particles (median 183), well under the legend's "about 1000
synaptosomes per area", and one image in each folder has nothing to measure. The re-implementation does not
reproduce the paper's segmentation closely (it is not FIJI, and the threshold and perimeter are approximations), and
part B of the claim is read with that in mind.

The OME metadata of 165 images give excitation at 488, 561 and 640 nm for channels 0, 1 and 2, the order the archive
annotates (bassoon, TH, VAMP2). Three images of GS mouse 4 in the BSN-area folder (BSN2, BSN3 and BSN4) give 488 nm
for all three channels; they are read in the same order as the others. Before the size and circularity limits, the
Default threshold gives a median of 1186 VAMP2 objects per image, near the legend's "about 1000"; the size limit of
0.04 to 0.40 um2 alone leaves 193, and the circularity limit 183. The figure shows that the kept particles are the
brighter puncta and that many dimmer puncta fall below the threshold. Which step differs from the paper's FIJI run
cannot be told from these outputs.

## Claims

### source-data-link

> The dandiset's recordings are those behind the paper's Figure 6 and Supplementary Figures 7 and 8, and the paper's
> published per-stimulation traces, evoked integrals and spontaneous transient integrals can be recomputed from them.

Records are matched by mouse ID, group, labelled power and the rank of the stimulation within its power. The source
data file is read in memory from Zenodo.

In [ ]:
md5, src = lrrk2.load_source_data()
print("Source data file md5:", md5, "; matches the Zenodo record:", md5 == lrrk2.SOURCE_DATA_MD5)
print("Records per table:", {k: len(v) for k, v in src.items()})
line_of = {"6C": "Anxa1", "6E": "Anxa1", "6F": "Anxa1", "S7B": "Anxa1", "S7C": "Anxa1",
           "S8C": "Calb1", "S8D": "Calb1", "S8E": "Calb1"}

# (1) Mice
mice = {(k, d["mouse"], d["group"]) for k, v in src.items() for d in v}
part1 = pd.DataFrame([{"table": k, "mouse": m, "group": g, "in_dandiset": m in by_mouse,
                       "same_group": m in by_mouse and by_mouse[m]["group"] == g,
                       "same_line": m in by_mouse and by_mouse[m]["line"] == line_of[k]} for k, m, g in mice])
part1["match"] = part1.in_dandiset & part1.same_group & part1.same_line
mouse_level = part1.groupby("mouse").match.all()
print(f"\n(1) Mice: {int(mouse_level.sum())} of {len(mouse_level)} source-data mice are in the dandiset with the same genotype and line;",
      "dandiset mice not in the source data:", sorted(set(by_mouse) - set(mouse_level.index)))


def rank_match(records):
    """Pair each source record with the dandiset epoch of the same mouse and power and the same rank."""
    seen, out = {}, []
    for d in records:
        key = (d["mouse"], d["power"])
        r = seen.get(key, 0)
        seen[key] = r + 1
        s = by_mouse[d["mouse"]]
        idx = np.where(np.isclose(s["power_in_mW"], d["power"]))[0]
        out.append((d, s, idx[r], idx))
    return out


# (2) Triggered traces
rows = []
for key in ("6C", "S7B", "S8D"):
    for d, s, j, idx in rank_match(src[key]):
        ours = lrrk2.code_chunk(s, j)
        rows.append({"table": key, "line": s["line"], "mouse": d["mouse"], "power": d["power"],
                     "max_diff": np.nanmax(np.abs(ours - d["trace"])), "nan_here": int(np.isnan(ours).sum()),
                     "r": np.corrcoef(ours[~np.isnan(ours)], d["trace"][~np.isnan(ours)])[0, 1],
                     "slope": np.polyfit(ours[~np.isnan(ours)], d["trace"][~np.isnan(ours)], 1)[0],
                     "offset": np.polyfit(ours[~np.isnan(ours)], d["trace"][~np.isnan(ours)], 1)[1]})
part2 = pd.DataFrame(rows)
part2["match"] = part2.max_diff < 1e-6
print("\n(2) Triggered traces equal to the dandiset's dF/F from 1 s before the onset of the epoch with the same power and rank:")
print(part2.groupby("table").agg(records=("match", "size"), matched=("match", "sum"), largest_diff=("max_diff", "max"),
                                 median_r=("r", "median"), slope_min=("slope", "min"), slope_max=("slope", "max")).to_string())
print("Largest difference among matched traces, per table:", {k: f"{v:.2g}" for k, v in part2[part2.match].groupby("table").max_diff.max().items()})
print("Unmatched Anxa1 traces:")
print(part2[(part2.line == "Anxa1") & ~part2.match].to_string(index=False))
m4398 = by_mouse[4398]
print(f"Mouse 4398: last onset at {m4398['start_time'][-1]:.2f} s, processed series ends at {m4398['t'][-1]:.2f} s")

# (3) Evoked integrals
code_auc = {s["mouse"]: lrrk2.evoked_integrals(s, "code") * lrrk2.FS for s in sessions}  # in samples, as the code
rows = []
for key in ("6E", "S7C", "S8E"):
    for d, s, j, idx in rank_match(src[key]):
        four = np.where(np.isclose(s["power_in_mW"], 4.0))[0][list(idx).index(j)]
        rows.append({"table": key, "line": s["line"], "mouse": d["mouse"], "power": d["power"], "published": d["value"],
                     "ours": code_auc[d["mouse"]][j], "ours_4mW_same_rank": code_auc[d["mouse"]][four]})
part3 = pd.DataFrame(rows)
factor = float(np.nanmedian(part3.published / part3.ours))
part3["match"] = np.abs(part3.published - factor * part3.ours) <= 1e-4 * np.abs(part3.published)
part3["match_if_4mW"] = np.abs(part3.published - factor * part3.ours_4mW_same_rank) <= 1e-4 * np.abs(part3.published)
print(f"\n(3) Evoked integrals: common factor published / ours (code's window, in samples) = {factor:.6g}")
print(part3.groupby(["table", "power"]).agg(records=("match", "size"), matched=("match", "sum"),
                                            matched_if_4mW=("match_if_4mW", "sum")).to_string())
print("Calb1 records: ratio published / ours from", part3[part3.line == "Calb1"].eval("published / ours").min().round(6),
      "to", part3[part3.line == "Calb1"].eval("published / ours").max().round(6))
print("Unmatched Anxa1 records:", part3[(part3.line == "Anxa1") & ~part3.match][["table", "mouse", "power", "published", "ours"]].to_dict("records"))

# (4) Spontaneous transients
measures = {ex: {s["mouse"]: lrrk2.spontaneous_transients(s, ex) for s in sessions} for ex in ("methods", "code_as_written")}
rows = []
for key in ("6F", "S8C"):
    pub = pd.DataFrame(src[key])
    for m, g in pub.groupby("mouse", sort=False):
        row = {"table": key, "line": line_of[key], "mouse": m, "published": len(g)}
        for ex in ("methods", "code_as_written"):
            auc, peaks, means, per_duration = measures[ex][m]
            row[f"n_{ex}"] = len(auc)
            for name, v in (("integral", auc), ("mean", means), ("per_duration", per_duration)):
                ratio = g.value.values / v if len(v) == len(g) else np.array([np.nan])
                row[f"{name}_ratio_{ex}"] = np.median(ratio)
                row[f"{name}_ratio_cv_{ex}"] = np.std(ratio) / np.mean(ratio)
        rows.append(row)
part4 = pd.DataFrame(rows)
for ex in ("methods", "code_as_written"):
    for name in ("integral", "mean", "per_duration"):
        f4 = np.nanmedian(part4[f"{name}_ratio_{ex}"])
        for rtol, tag in ((1e-2, ""), (1e-4, "_1e-4")):
            ok = []
            for _, r in part4.iterrows():
                pub = pd.DataFrame(src[r.table]).query("mouse == @r.mouse").value.values
                v = measures[ex][r.mouse][{"integral": 0, "mean": 2, "per_duration": 3}[name]]
                ok.append(len(pub) == len(v) and bool(np.allclose(np.sort(pub), f4 * np.sort(v), rtol=rtol, atol=0)))
            part4[f"match_{name}_{ex}{tag}"] = ok
            print(f"(4) exclusion '{ex}', {name} of each transient: common factor {f4:.6g}; mice whose count and sorted values match",
                  f"within a relative {rtol:g}:", int(np.sum(ok)), "of", len(ok), "(Anxa1", int(np.sum(np.array(ok)[part4.line == 'Anxa1'])), "of 15)")
cols = ["table", "mouse", "published", "n_methods", "n_code_as_written", "integral_ratio_cv_methods", "mean_ratio_methods",
        "mean_ratio_cv_methods", "match_integral_methods", "match_mean_methods", "match_mean_methods_1e-4",
        "match_mean_code_as_written"]
print(part4[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for _, r in part4[(part4.line == "Anxa1") & ~part4.match_mean_methods].iterrows():
    pub = np.sort(pd.DataFrame(src[r.table]).query("mouse == @r.mouse").value.values)
    v = np.sort(measures["methods"][r.mouse][2]) * float(np.nanmedian(part4.mean_ratio_methods))
    rel = np.abs(pub - v) / np.abs(pub)
    print(f"Anxa1 mouse {r.mouse} fails the 1% match on the mean: {int((rel > 1e-2).sum())} of {len(rel)} sorted values differ by more",
          f"than 1% (largest {rel.max():.3g}); its own ratio {r.mean_ratio_methods:.6g}, CV {r.mean_ratio_cv_methods:.3g}")
print("Transients: published", part4.published.sum(), "; here, Methods exclusion", part4.n_methods.sum(),
      "; code's exclusion as written", part4.n_code_as_written.sum())
print("Count equal to the published count, Methods exclusion:", part4.groupby("line").apply(lambda g: f"{int((g.published == g.n_methods).sum())} of {len(g)}").to_dict())

score = {"mice": float(mouse_level.mean()), "traces": float(part2.match.mean()), "evoked integrals": float(part3.match.mean()),
         "spontaneous integrals (best window)": float(max(part4.match_integral_methods.mean(), part4.match_integral_code_as_written.mean()))}
print("Part (4) with integrals at a relative 1e-4, the tolerance of part (3):",
      round(float(max(part4["match_integral_methods_1e-4"].mean(), part4["match_integral_code_as_written_1e-4"].mean())), 3))
print("\nFraction of records that match, per part, as the criterion scores them:", {k: round(v, 3) for k, v in score.items()})
print("Variant, part (4) with the mean dF/F of each transient, within 1%:",
      round(float(max(part4.match_mean_methods.mean(), part4.match_mean_code_as_written.mean())), 3), "; within 1e-4:",
      round(float(max(part4["match_mean_methods_1e-4"].mean(), part4["match_mean_code_as_written_1e-4"].mean())), 3))

# (4b) Which measure the published values are: for each candidate, the spread of published / ours within each mouse
# (transients in time order, Methods exclusion), and the mice that match at a relative 1e-4.
rows = []
for name in ("integral", "mean", "per_duration"):
    for line, g in part4.groupby("line"):
        rows.append({"measure": name, "line": line, "common_factor": float(np.nanmedian(part4[f"{name}_ratio_methods"])),
                     "largest_within_mouse_cv": float(g[f"{name}_ratio_cv_methods"].max()),
                     "median_within_mouse_cv": float(g[f"{name}_ratio_cv_methods"].median()),
                     "mice_matched_1e-4": f"{int(g[f'match_{name}_methods_1e-4'].sum())} of {len(g)}",
                     "mice_matched_1%": f"{int(g[f'match_{name}_methods'].sum())} of {len(g)}"})
print("\n(4b) Candidate measures of the published spontaneous values (per_duration: the trapezoid integral divided by the",
      "number of samples in the transient, % dF/F):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.6g}"))
score_pd = dict(score, **{"spontaneous integrals (best window)": float(max(part4["match_per_duration_methods_1e-4"].mean(),
                                                                          part4["match_per_duration_code_as_written_1e-4"].mean()))})
print("Variant, part (4) with the integral divided by the duration, at 1e-4; the four parts:", {k: round(v, 3) for k, v in score_pd.items()})

# (6) The paper's statistics recomputed from its own published values, to check the source data against the text.
pubdf = {k: pd.DataFrame(src[k]) for k in ("6E", "6F", "S7C", "S8C", "S8E")}
rows = []
for key, power, paper_p in (("6E", 0.5, "1.10e-20"), ("S7C", 4.0, "2.39e-16"), ("S8E", 0.5, "0.0002"), ("S8E", 4.0, "0.001")):
    g = pubdf[key][pubdf[key].power == power]
    gs, wt = g[g.group == "GS"].value, g[g.group == "WT"].value
    rows.append({"panel": key, "power": power, "n_GS": len(gs), "n_WT": len(wt), "p_one_tailed": lrrk2.one_tailed_less(gs, wt)[1],
                 "paper_p": paper_p, "cohens_d": lrrk2.cohens_d(wt, gs)})
for key, paper_p in (("6F", "2.75e-177"), ("S8C", "0.98")):
    g = pubdf[key]
    gs, wt = g[g.group == "GS"].value, g[g.group == "WT"].value
    rows.append({"panel": key, "power": np.nan, "n_GS": len(gs), "n_WT": len(wt), "p_one_tailed": lrrk2.one_tailed_less(gs, wt)[1],
                 "paper_p": paper_p, "cohens_d": lrrk2.cohens_d(wt, gs)})
print("\n(6) Statistics from the published values themselves (paper's Cohen's d at 0.5 mW: Anxa1 2.05, Calb1 0.76):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))

To see why the Calb1 records differ, the dF/F is recomputed from the raw 470 nm signal in the dandiset with the
code's binning and baseline correction, at the code's background fraction (0.85 of the sliding 8th percentile) and at
other fractions, and compared with the published triggered traces. The edges of the sliding percentile are not
reproduced exactly, so the comparison with the stored series leaves out the first and last 30 s.

In [ ]:
# (5) Why the Calb1 records differ: dF/F recomputed from the raw 470 nm signal in the dandiset with the code's
# processing, and with other background fractions, against the published triggered traces.
rows = []
for s in sessions:
    binned = lrrk2.raw_binned(s)
    key = "6C" if s["line"] == "Anxa1" else "S8D"
    pairs = [(d, j) for d, ss, j, idx in rank_match([d for d in src[key] if d["mouse"] == s["mouse"]])]
    stored_diff = np.abs(lrrk2.dff_from_raw(binned) - s["dff"])[3000:-3000].max()
    best = None
    for frac in np.round(np.arange(0.0, 0.951, 0.05), 2):
        dff = lrrk2.dff_from_raw(binned, frac)
        err = np.median([np.abs(dff[s["onset"][j] - 100:s["onset"][j] + 150] - d["trace"]).max() for d, j in pairs])
        if frac == 0.85:
            at_085 = err
        if best is None or err < best[1]:
            best = (frac, err)
    rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "stored_vs_recomputed_max_diff": stored_diff,
                 "median_diff_at_0.85": at_085, "best_fraction": best[0], "median_diff_at_best": best[1]})
raw_check = pd.DataFrame(rows).sort_values(["line", "group", "mouse"])
print("Largest difference between the stored dF/F and dF/F recomputed from raw with the code's 0.85 (away from the edges),",
      "and the median over a mouse's published traces of the largest difference to dF/F recomputed with background fraction",
      "0.85 and with the best fraction of 0 to 0.95 (% dF/F):")
print(raw_check.round(3).to_string(index=False))

In [ ]:
# (5b) Is the Calb1 mismatch one of epoch or power labels, or of processing? Per mouse, the fit of the published S8D
# traces on the dandiset's matched traces (published = slope x ours + offset), and for each published trace the
# dandiset epoch, over all 64 epochs and all 8 powers of that mouse, that it is closest to.
s8d = part2[part2.table == "S8D"]
print("S8D traces against the matched dandiset epoch, per mouse (r, slope and offset of published on ours; offset in % dF/F):")
print(s8d.groupby("mouse").agg(traces=("r", "size"), r_median=("r", "median"), r_min=("r", "min"), slope_median=("slope", "median"),
                               slope_min=("slope", "min"), slope_max=("slope", "max"), offset_median=("offset", "median"),
                               max_diff_median=("max_diff", "median")).round(3).to_string())
rows = []
for d, s, j, idx in rank_match(src["S8D"]):
    chunks = np.array([lrrk2.code_chunk(s, k) for k in range(len(s["onset"]))])
    ok = ~np.isnan(chunks).any(1)
    diff = np.where(ok, np.nanmax(np.abs(chunks - d["trace"]), 1), np.inf)
    r = np.array([np.corrcoef(c, d["trace"])[0, 1] if o else -np.inf for c, o in zip(chunks, ok)])
    kd, kr = int(np.argmin(diff)), int(np.argmax(r))
    rows.append({"mouse": d["mouse"], "power": d["power"], "matched_epoch": j, "diff_matched": diff[j], "r_matched": r[j],
                 "best_epoch_by_diff": kd, "best_diff": diff[kd], "best_diff_power": s["power_in_mW"][kd],
                 "best_epoch_by_r": kr, "best_r": r[kr], "best_r_power": s["power_in_mW"][kr]})
best = pd.DataFrame(rows)
print("\nFor the 384 S8D traces: best epoch by largest absolute difference and by correlation, over all epochs of the mouse")
print("best epoch by difference is the matched one:", int((best.best_epoch_by_diff == best.matched_epoch).sum()), "of", len(best),
      "; has the labelled power:", int((best.best_diff_power == best.power).sum()))
print("best epoch by correlation is the matched one:", int((best.best_epoch_by_r == best.matched_epoch).sum()), "of", len(best),
      "; has the labelled power:", int((best.best_r_power == best.power).sum()))
print("traces with any epoch within 1e-6:", int((best.best_diff < 1e-6).sum()), "; best difference over all epochs (% dF/F), median",
      round(best.best_diff.median(), 2), "range", round(best.best_diff.min(), 2), "to", round(best.best_diff.max(), 2))
print(best.groupby("mouse").agg(traces=("power", "size"), best_by_r_is_matched=("best_epoch_by_r", lambda v: int((v == best.loc[v.index, "matched_epoch"]).sum())),
                                diff_matched_median=("diff_matched", "median"), best_diff_median=("best_diff", "median"),
                                r_matched_median=("r_matched", "median"), best_r_median=("best_r", "median")).round(3).to_string())


**Result: not reproduced.** All 27 mice in the source data are in the dandiset with the same genotype and line. For
Anxa1 the link holds: 120 of 120 triggered traces of Figure 6C and 479 of 480 of Supplementary Figure 7B equal the
dandiset's dF/F from 1 s before onset (largest difference among matched traces 5.7e-13 in each table), the published evoked integrals equal
the code's measure times 0.0001 for all Anxa1 records except one (119 of 120 at 4 mW), and the transient counts match
for 15 of 15 mice. The two Anxa1 misses are the last 4 mW stimulation of mouse 4398, whose processed series ends 0.8 s
after onset. For Calb1 none of the 384 traces and none of the 384 integrals match, and the transient counts match for
1 of 12 mice. dF/F recomputed from the raw signal with the code's processing reproduces the stored Calb1 series (largest
difference 1.47 % dF/F) but not the published traces (median differences 66 to 519 % dF/F at the code's background
fraction, and 22 to 88 at the best fraction for each mouse). The published spontaneous values are not integrals:
they are 0.005 times the trapezoid integral divided by the number of samples in the transient (part 4b: within-mouse
spread of the ratio at most 2.2e-15, 15 of 15 Anxa1 mice matched at 1e-4). Part (4) as the criterion fixed it, with
integrals, matches for no mouse. Scored as the criterion sets out, the four parts match for 1.0, 0.609, 0.609 and 0
of their records, so the outcome is not reproduced. With the published measure in part (4) the parts give 1.0, 0.609,
0.609 and 0.556 (15 of 27 mice, no Calb1 mouse), which would be partially reproduced; the criterion fixed integrals,
so this is a variant. The Figure 6E integrals match the 0.5 mW epochs (120 of 120) and none of the 4 mW ones, so the
figure uses the power its legend states. The published values reproduce the paper's statistics (Figure 6E
p = 1.1e-20, Figure 6F p = 2.75e-177, Supplementary Figure 8C p = 0.989, Cohen's d 2.06 and 0.765).

Part (4) scores 0 with integrals at either tolerance, 1% or the 1e-4 of part (3). The mean dF/F over each transient
comes close (13 of 15 Anxa1 mice within 1%, none within 1e-4, per-mouse factor 0.004994 to 0.004999), as it differs
from the published measure only in weighting the two end samples.

The Calb1 mismatch is not one of epoch or power labels. Against the matched epochs the published S8D traces have
per-mouse median correlations of 0.76 to 0.99, median slopes of 0.21 to 0.75 and median offsets of -13.8 to 16.2
% dF/F. Searched over all 64 epochs and all 8 powers of each mouse, no epoch is within 1e-6 of any of the 384 traces;
the closest differs by 15.7 to 150.8 % dF/F (median 47.1). Correlation does not single out an epoch: the epoch with
the highest correlation is the matched one for 7 of 384 traces and has the labelled power for 86. The published Calb1
traces are smaller and differently shaped versions of the dandiset's responses, which points to processing; the
dandiset does not hold that processing.

### Evoked integrals

Integrals for every train under the four windows: the Methods' light-on period (main), the code's -0.1 to 1.0 s
(`code`), the window `chunking_index.m` computes as written, starting 1 s after onset (`code_as_written`), and a fixed
0 to 0.5 s (`fixed`). All use the mean from 1.0 to 0.1 s before onset as baseline (shifted with the window for
`code_as_written`).

In [ ]:
def evoked_table(window):
    rows = []
    for s in sessions:
        values = lrrk2.evoked_integrals(s, window)
        for j, v in enumerate(values):
            rows.append({"mouse": s["mouse"], "line": s["line"], "group": s["group"], "power": s["power_in_mW"][j],
                         "train_s": round(s["stop_time"][j] - s["start_time"][j], 2), "value": v})
    return pd.DataFrame(rows)


windows = ["methods", "code", "code_as_written", "fixed"]
evoked = {w: evoked_table(w) for w in windows}
print("Evoked integrals (% dF/F x s) that could not be computed (window outside the recording):",
      {w: int(evoked[w].value.isna().sum()) for w in windows})


def group_test(df):
    gs, wt = df[df.group == "GS"].value.dropna(), df[df.group == "WT"].value.dropna()
    t, p = lrrk2.one_tailed_less(gs, wt)
    return {"n_GS": len(gs), "n_WT": len(wt), "mean_GS": gs.mean(), "mean_WT": wt.mean(),
            "t": t, "p_one_tailed": p, "cohens_d": lrrk2.cohens_d(wt, gs)}


def per_mouse_tests(df, alpha):
    wt = df[df.group == "WT"].value.dropna()
    rows = []
    for m, g in df[df.group == "GS"].groupby("mouse"):
        v = g.value.dropna()
        row = {"mouse": m, "n": len(v), "mean": v.mean(), "WT_mean": wt.mean()}
        for name, equal_var, tails in (("a_welch_1t", False, 1), ("b_welch_2t", False, 2),
                                       ("c_student_2t", True, 2), ("d_student_1t", True, 1)):
            p = (lrrk2.one_tailed_less(v, wt, equal_var)[1] if tails == 1 else lrrk2.two_tailed(v, wt, equal_var)[1])
            row[f"p_{name}"] = p
            row[f"sig_{name}"] = bool(p < alpha and v.mean() < wt.mean())
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# Check of evoked_integrals on a synthetic series that ends 0.81 s after the last onset, as mouse 4398's does: the
# light-on window (0.51 s) and the 0 to 0.5 s window fit and are computed; the code's window (to 1.0 s) does not.
toy = {"dff": np.r_[np.zeros(200), np.ones(81)], "onset": np.array([100, 200]), "offset": np.array([150, 251])}
toy_vals = {w: lrrk2.evoked_integrals(toy, w) for w in ("methods", "fixed", "code")}
print("Synthetic check, integrals of the second train per window:", {w: v[1] for w, v in toy_vals.items()})
assert np.isclose(toy_vals["methods"][1], 0.51) and np.isclose(toy_vals["fixed"][1], 0.5) and np.isnan(toy_vals["code"][1])


### fig6e-anxa1-evoked-reduced

> Optogenetically evoked dopamine release in the dorsolateral striatum (Anxa1+ projection region) at 0.5 mW is lower in
> LRRK2-G2019S than in LRRK2-WT mice (64 against 56 stimulations, p = 1.10e-20, one-tailed two-sample t-test).

The plot shows the mean dF/F (shading: SEM over stimulations) at 0.5 mW, with the 405 nm control dashed, and the
light-on period shaded. All points are within the axes; the ranges are printed.

In [ ]:
anxa = evoked["methods"].query("line == 'Anxa1' and power == 0.5")
main = group_test(anxa)
print("Anxa1, 0.5 mW, Methods window (light-on period):")
print({k: (float(f"{v:.3g}") if isinstance(v, float) else v) for k, v in main.items()})
print(f"one-tailed p = {main['p_one_tailed']:.3g} (paper 1.10e-20)")
print("\nPer mouse (mean and range of the eight integrals, train length):")
print(anxa.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), min=("value", "min"),
                                           max=("value", "max"), train_s=("train_s", "first")).round(2).to_string())
print("\nVariants (window, power): counts, means, one-tailed p, Cohen's d")
rows = []
for w in windows:
    for q in (0.5, 4.0):
        r = group_test(evoked[w].query("line == 'Anxa1' and power == @q"))
        rows.append({"window": w, "power": q, **r})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nVariant without the two G2019S mice whose 0.5 mW trains are 0.33 s long (3742, 4885), Methods window:")
r = group_test(anxa[~anxa.mouse.isin([3742, 4885])])
print({k: (f"{v:.3g}" if isinstance(v, float) else v) for k, v in r.items()})
print("Variant with one value per mouse (mouse means), Methods window: Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(anxa[anxa.group == 'GS'].groupby('mouse').value.mean(), anxa[anxa.group == 'WT'].groupby('mouse').value.mean(), False)[1]:.3g}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
tt = np.arange(-100, 150) / lrrk2.FS
for ax, line in zip(axes, ("Anxa1", "Calb1")):
    for group, colour in (("GS", "tab:blue"), ("WT", "tab:gray")):
        tr, iso = [], []
        for s in sessions:
            if s["line"] != line or s["group"] != group:
                continue
            for j in np.where(s["power_in_mW"] == 0.5)[0]:
                tr.append(lrrk2.code_chunk(s, j))
                on = s["onset"][j]
                iso.append(s["dff405"][on - 100:on + 150])
        tr, iso = np.array(tr), np.array(iso)
        m, se = tr.mean(0), tr.std(0, ddof=1) / np.sqrt(len(tr))
        ax.plot(tt, m, color=colour, label=f"{group} (n = {len(tr)})")
        ax.fill_between(tt, m - se, m + se, color=colour, alpha=0.3)
        ax.plot(tt, iso.mean(0), color=colour, ls="--", lw=1)
        print(f"{line} {group}: mean trace from {m.min():.1f} to {m.max():.1f} % dF/F; 405 nm mean from {iso.mean(0).min():.1f} to {iso.mean(0).max():.1f}")
    ax.axvspan(0, 0.5, color="red", alpha=0.1)
    ax.set_title(f"{line}, 0.5 mW (dashed: 405 nm)")
    ax.set_xlabel("time from onset (s)")
    ax.set_ylabel("dF/F (%)")
    ax.legend(frameon=False)
plt.tight_layout()
plt.show()

**Result: reproduced.** At 0.5 mW there are 64 GS and 56 WT stimulations, as in the paper. The mean integral over the
light-on period is 48.8 against 128.0 % dF/F x s, one-tailed p = 1.16e-22 (paper 1.10e-20), Cohen's d 2.21. With the
code's window p = 1.1e-20, the paper's value. The reduction holds at 4 mW (p = 4.08e-18), without the two GS mice given
0.33 s trains (p = 4.16e-17), and with one value per mouse (Welch p = 0.000434). The window `chunking_index.m` computes
as written reverses the sign (one-tailed p = 1, means -11.8 and -33), as it starts 1 s after onset. Every GS mouse
except 3743 has a lower mean than every WT mouse (GS 3.28 to 82.33 without 3743, WT 90.86 to 190.21).

### fig6e-anxa1-evoked-per-mouse

> Seven of eight individual G2019S mice show significantly reduced evoked dopamine release at 0.5 mW compared with the
> WT group; mouse 3743 does not.

In [ ]:
pm = per_mouse_tests(anxa, 0.05 / 8)
print("Each Anxa1 G2019S mouse against all WT integrals, 0.5 mW, Methods window; significant when p < 0.05/8 and the mean is lower.")
print(pm.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pm[c].sum()) for c in pm.columns if c.startswith("sig_")})
print("Not significant under each reading:", {c[4:]: pm.loc[~pm[c], "mouse"].tolist() for c in pm.columns if c.startswith("sig_")})
print("Paper's p-values (legend): 4885 2.57e-10, 4651 0.0004, 4652 7.36e-05, 3742 4.14e-15, 3743 0.478, 4004 2.81e-13, 4005 1.72e-14, 4007 2.36e-12")
print("\nVariants (other windows, and 4 mW): significant mice per reading")
for w in windows:
    for q in (0.5, 4.0):
        v = per_mouse_tests(evoked[w].query("line == 'Anxa1' and power == @q"), 0.05 / 8)
        print(f"  {w:16s} {q} mW:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
              "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())

**Result: reproduced.** Under all four readings 7 of 8 GS mice are below the WT group at p < 0.05/8, and 3743 is the
exception (mean 125 against 128, Welch one-tailed p = 0.362), as in the paper. The Student one-tailed p-values are of
the same order as the legend's, for example 1.01e-04 against 4.0e-04 for 4651 and 3.73e-16 against 4.14e-15 for 3742.
With the code's window the count is 7 under every reading, and at 4 mW it is 7 under the Welch readings. Two of the
seven, 3742 and 4885, received their paper powers as 0.33 s trains, unlike every WT mouse, and the data cannot confirm
their power labels (data check); the other five significant mice are 4004, 4005, 4007, 4651 and 4652.

### supp7c-anxa1-evoked-all-powers

> The reduction of evoked dopamine release in Anxa1 G2019S mice holds at each of the four stimulation powers, 0.1, 0.5,
> 1 and 4 mW (p from 1.29e-15 to 1.10e-20).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        rows.append({"window": w, "power": q, **group_test(evoked[w].query("line == 'Anxa1' and power == @q"))})
s7 = pd.DataFrame(rows)
print("Anxa1, evoked integrals per power; paper p: 0.1 mW 1.29e-15, 0.5 mW 1.10e-20, 1 mW 1.22e-17, 4 mW 2.39e-16")
print(s7.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
main7 = s7[s7.window == "methods"]
print("\nMethods window: powers with G2019S lower at one-tailed p < 0.05:",
      int(((main7.p_one_tailed < 0.05) & (main7.mean_GS < main7.mean_WT)).sum()), "of 4")

**Result: reproduced.** GS integrals are lower at all four powers, one-tailed p from 3.13e-15 (0.1 mW) to 1.16e-22
(0.5 mW) with the Methods' window. With the code's window the p-values at 0.1, 0.5 and 1 mW are 1.29e-15, 1.1e-20 and
1.22e-17, the paper's values; at 4 mW it is 5.62e-16 against 2.39e-16, as the code's window, which runs to 1.0 s after
onset, leaves 55 WT stimulations in place of 56. The Methods window fits the last 4 mW train of mouse 4398 and keeps
all 56.

### Spontaneous transients

Transients for every mouse, with the Methods' exclusion of 1 s before to 1.5 s after each onset (main) and with the
window the code removes as written (1 s to 3.5 s after onset). Each transient's integral, peak and mean dF/F are kept.

In [ ]:
def spont_table(exclusion):
    rows = []
    for s in sessions:
        auc, peaks, means, per_duration = lrrk2.spontaneous_transients(s, exclusion)
        rows += [{"mouse": s["mouse"], "line": s["line"], "group": s["group"], "value": a, "peak": p, "mean": m, "per_duration": d}
                 for a, p, m, d in zip(auc, peaks, means, per_duration)]
    return pd.DataFrame(rows)


spont = {ex: spont_table(ex) for ex in ("methods", "code_as_written")}
for ex, df in spont.items():
    print(f"{ex}: {len(df)} transients over {df.mouse.nunique()} mice; integral (% dF/F x s) from {df.value.min():.2f} to {df.value.max():.1f}")

### fig6f-anxa1-spontaneous-reduced

> Spontaneous dopamine transients in the dorsolateral striatum are smaller in G2019S than in WT mice (1064 against 798
> transients, p = 2.75e-177, one-tailed two-sample t-test), with 57 to 285 transients per mouse.

In [ ]:
sa = spont["methods"].query("line == 'Anxa1'")
main_s = group_test(sa)
print("Anxa1 spontaneous transients, Methods exclusion:")
print({k: (float(f"{v:.3g}") if isinstance(v, float) else v) for k, v in main_s.items()})
print(f"one-tailed p = {main_s['p_one_tailed']:.3g} (paper 2.75e-177; counts 1064 G2019S, 798 WT)")
per = sa.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), median=("value", "median"),
                                          max=("value", "max"), mean_peak=("peak", "mean"))
print("\nPer mouse:")
print(per.round(2).to_string())
print("Transients per mouse from", per.n.min(), "to", per.n.max(), "(paper 57 to 285; criterion 51.3 to 313.5)")
v = spont["code_as_written"].query("line == 'Anxa1'")
r = group_test(v)
print("\nVariant, the code's exclusion window as written:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r.items()})
print("Variant, one value per mouse (mouse means), Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(per.loc['GS', 'mean'], per.loc['WT', 'mean'], False)[1]:.3g}")
print("Variant, transient peaks in place of integrals: one-tailed p =",
      f"{lrrk2.one_tailed_less(sa[sa.group == 'GS'].peak, sa[sa.group == 'WT'].peak)[1]:.3g}")
r = group_test(sa.assign(value=sa["mean"]))
print("Variant, the mean dF/F over each transient (close to the published values, see source-data-link):",
      {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r.items()})
r = group_test(sa.assign(value=0.005 * sa["per_duration"]))
print("Variant, 0.005 times the integral divided by the duration (the published values, see source-data-link (4b)):",
      {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r.items()})
print(f"   one-tailed p = {r['p_one_tailed']:.4g} (paper 2.75e-177)")


**Result: reproduced.** The Methods' detection gives 1064 GS and 798 WT transients, the paper's counts, with 57 to 285
per mouse, the paper's range. Mean integrals are 131.9 against 302.9 % dF/F x s, one-tailed p = 4.85e-72 (paper
2.75e-177). The published values are 0.005 times the integral divided by the duration of each transient (see
`source-data-link`, part 4b); with that measure p = 2.747e-177, the paper's value. With one value per mouse the
difference gives Welch p = 0.0318, and the code's exclusion window as written gives p = 1.3e-68 on 648 and 363
transients.

### fig6f-anxa1-spontaneous-per-mouse

> Six of eight individual G2019S mice show significantly smaller spontaneous dopamine transients than the WT group;
> mice 4651 and 3743 do not.

In [ ]:
pms = per_mouse_tests(sa, 0.05 / 8)
print("Each Anxa1 G2019S mouse's transient integrals against all WT transients, Methods exclusion; threshold 0.05/8.")
print(pms.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pms[c].sum()) for c in pms.columns if c.startswith("sig_")})
print("Not significant under each reading:", {c[4:]: pms.loc[~pms[c], "mouse"].tolist() for c in pms.columns if c.startswith("sig_")})
print("Paper's p-values (legend): 4885 1.44e-49, 4651 0.008, 4652 6.21e-13, 3742 5.28e-88, 3743 0.97, 4004 1.24e-95, 4005 4.13e-109, 4007 5.00e-107")
v = per_mouse_tests(sa.assign(value=sa["mean"]), 0.05 / 8)
print("\nVariant, the mean dF/F over each transient:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
      "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())
print(v[["mouse", "mean", "WT_mean", "p_a_welch_1t", "p_d_student_1t"]].to_string(index=False, float_format=lambda x: f"{x:.3g}"))
v = per_mouse_tests(sa.assign(value=0.005 * sa["per_duration"]), 0.05 / 8)
print("\nVariant, 0.005 times the integral divided by the duration (the published values):",
      {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
      "not significant (d, Student one-tailed):", v.loc[~v.sig_d_student_1t, "mouse"].tolist())
print(v[["mouse", "mean", "WT_mean", "p_a_welch_1t", "p_d_student_1t"]].to_string(index=False, float_format=lambda x: f"{x:.3g}"))
v = per_mouse_tests(spont["code_as_written"].query("line == 'Anxa1'"), 0.05 / 8)
print("\nVariant, code's exclusion as written:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")},
      "not significant (a):", v.loc[~v.sig_a_welch_1t, "mouse"].tolist())

**Result: partially reproduced.** By the integral, the measure the criterion fixed, 5 of 8 GS mice are below the WT
group under every reading. The exceptions are 3743 (Welch one-tailed p = 0.431), 4651, whose mean (417) is above the
WT mean (303), and 4652 (p = 0.0128). With the published measure, 0.005 times the integral divided by the duration,
reading (d), the one-tailed Student test of the legend, gives 6 of 8 with 3743 and 4651 as the exceptions, and the
legend's p-values (3742 5.28e-88, 3743 0.965 against 0.97, 4004 1.24e-95, 4005 4.13e-109, 4007 5e-107, 4651 0.00844
against 0.008, 4652 6.21e-13, 4885 1.44e-49). The Welch readings give 7 there, as 4651 reaches p = 1.61e-05, so the
paper's count follows the one-tailed Student test of its legend, not the Welch test of its Methods. The difference
from the paper comes from the measure, not the data.

### supp8c-calb1-spontaneous-not-reduced

> In the dorsomedial striatum (Calb1+ projection region), spontaneous dopamine transients are not reduced in G2019S
> mice (970 against 682 transients, p = 0.98).

In [ ]:
sc = spont["methods"].query("line == 'Calb1'")
r = group_test(sc)
print("Calb1 spontaneous transients, Methods exclusion (paper: 970 G2019S, 682 WT, one-tailed p = 0.98):")
print({k: (float(f"{v:.4g}") if isinstance(v, float) else v) for k, v in r.items()})
print(f"one-tailed p (G2019S lower) = {r['p_one_tailed']:.3g}; difference of means (WT - G2019S) = {r['mean_WT'] - r['mean_GS']:.3f} % dF/F x s")
print("Two-tailed p =", f"{lrrk2.two_tailed(sc[sc.group == 'GS'].value, sc[sc.group == 'WT'].value)[1]:.3g}")
per = sc.groupby(["group", "mouse"]).agg(n=("value", "size"), mean=("value", "mean"), median=("value", "median"))
print("\nPer mouse:")
print(per.round(3).to_string())
v = spont["code_as_written"].query("line == 'Calb1'")
r2 = group_test(v)
print("\nVariant, the code's exclusion window as written:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r2.items()})
print("Variant, mouse means, Welch one-tailed p =",
      f"{lrrk2.one_tailed_less(per.loc['GS', 'mean'], per.loc['WT', 'mean'], False)[1]:.3g}")
r3 = group_test(sc.assign(value=sc["mean"]))
print("Variant, the mean dF/F over each transient:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r3.items()})
r4 = group_test(sc.assign(value=0.005 * sc["per_duration"]))
print("Variant, 0.005 times the integral divided by the duration:", {k: (f"{x:.3g}" if isinstance(x, float) else x) for k, x in r4.items()})

# How the main test moves with the exclusion window: 1 s before to 1.5 s after onset (100 and 150 samples), each end
# moved by up to 10 samples in steps of 2.
cal = [s for s in sessions if s["line"] == "Calb1"]
rows = []
for pre in range(90, 111, 2):
    for post in range(140, 161, 2):
        vals = {"GS": [], "WT": []}
        for s in cal:
            vals[s["group"]].append(lrrk2.spontaneous_transients(s, (pre, post))[0])
        gs, wt = np.concatenate(vals["GS"]), np.concatenate(vals["WT"])
        rows.append({"before": pre, "after": post, "n_GS": len(gs), "n_WT": len(wt), "p": lrrk2.one_tailed_less(gs, wt)[1]})
grid = pd.DataFrame(rows)
print("\nOne-tailed p (G2019S lower) by exclusion window, samples before (rows) and after (columns) onset:")
print(grid.pivot(index="before", columns="after", values="p").to_string(float_format=lambda v: f"{v:.3f}"))
print(f"p from {grid.p.min():.3g} to {grid.p.max():.3g}; below 0.05 in {int((grid.p < 0.05).sum())} of {len(grid)} windows;",
      f"transients GS {grid.n_GS.min()} to {grid.n_GS.max()}, WT {grid.n_WT.min()} to {grid.n_WT.max()}")


**Result: reproduced.** By the paper's test, GS transients are not significantly smaller: one-tailed p = 0.0509 on 896
GS and 635 WT transients (both within tolerance of 970 and 682), difference of means 21.1 % dF/F x s, Cohen's d 0.085.
The result sits at the threshold and is fragile: moving either end of the exclusion window by up to 10 samples gives
p from 0.0403 to 0.0792, below 0.05 in 59 of 121 windows. With the published measure (0.005 times the integral divided
by the duration) GS are lower at p = 2.45e-09 (d = 0.305), with the mean dF/F over each transient at p = 2.25e-09, and
with the code's exclusion window as written at p = 1.81e-05; with one value per mouse, Welch p = 0.402. The dandiset's
Calb1 dF/F is not the trace the paper analysed (see `source-data-link`), and the published values give p = 0.989.

### supp8e-calb1-evoked-reduced

> Evoked dopamine release in the dorsomedial striatum from Calb1+ neuron stimulation is reduced in G2019S mice at each
> of four powers (56 against 40 stimulations per power, p from 6.45e-05 to 0.001).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        rows.append({"window": w, "power": q, **group_test(evoked[w].query("line == 'Calb1' and power == @q"))})
s8 = pd.DataFrame(rows)
print("Calb1, evoked integrals per power; paper p: 0.1 mW 6.45e-05, 0.5 mW 0.0002, 1 mW 0.0001, 4 mW 0.001 (56 against 40)")
print(s8.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
main8 = s8[s8.window == "methods"]
print("\nMethods window: powers with G2019S lower at one-tailed p < 0.05:",
      int(((main8.p_one_tailed < 0.05) & (main8.mean_GS < main8.mean_WT)).sum()), "of 4")
cal = evoked["methods"].query("line == 'Calb1' and power == 0.5")
print("\nPer mouse, 0.5 mW, Methods window:")
print(cal.groupby(["group", "mouse"]).value.agg(["size", "mean", "min", "max"]).round(2).to_string())
print("\nVariant with one value per mouse (mouse means, 7 GS and 5 WT), Methods window, Welch one-tailed p per power:")
for q in lrrk2.PAPER_POWERS:
    mm = evoked["methods"].query("line == 'Calb1' and power == @q").groupby(["group", "mouse"]).value.mean()
    print(f"  {q} mW: p = {lrrk2.one_tailed_less(mm.loc['GS'], mm.loc['WT'], False)[1]:.3g}, mean of mouse means GS {mm.loc['GS'].mean():.1f}, WT {mm.loc['WT'].mean():.1f}")

**Result: reproduced.** There are 56 GS and 40 WT stimulations per power, as in the paper. GS are lower at all four
powers, one-tailed p from 4.38e-07 (4 mW) to 6.29e-10 (0.5 mW), smaller p-values than the paper's 6.45e-05 to 0.001;
the published integrals do not come from the dandiset's Calb1 dF/F (see `source-data-link`). With one value per mouse
(7 GS and 5 WT), Welch one-tailed p is 0.0424, 0.0413, 0.0726 and 0.0916 at 0.1, 0.5, 1 and 4 mW, so the reduction is
significant across mice at the two lower powers only.

### supp8e-calb1-few-mice-reduced

> Only 2 of 7 G2019S mice show reduced Calb1+ evoked dopamine release relative to the WT group.

In [ ]:
pmc = per_mouse_tests(cal, 0.05 / 7)
print("Each Calb1 G2019S mouse against all WT integrals, 0.5 mW, Methods window; threshold 0.05/7.")
print(pmc.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("\nSignificant mice per reading:", {c[4:]: int(pmc[c].sum()) for c in pmc.columns if c.startswith("sig_")})
print("Significant under each reading:", {c[4:]: pmc.loc[pmc[c], "mouse"].tolist() for c in pmc.columns if c.startswith("sig_")})
print("\nVariants (other powers and windows): significant mice per reading")
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        v = per_mouse_tests(evoked[w].query("line == 'Calb1' and power == @q"), 0.05 / 7)
        print(f"  {w:16s} {q} mW:", {c[4:]: int(v[c].sum()) for c in v.columns if c.startswith("sig_")})

**Result: partially reproduced.** At 0.5 mW, the number of GS mice below the WT group at p < 0.05/7 is 7 (Welch
one-tailed), 6 (Welch two-tailed) and 4 (Student, either tail), against the paper's 2; the closest reading gives 4.
Student tests give 2 at 0.1 and 1 mW and 1 or 3 at 4 mW, so the count depends on power and test. Mouse 740 (mean 17.3
against 131) is below the WT group under every reading.

### cohens-d-anxa1-larger

> The G2019S reduction of evoked release at 0.5 mW is much larger for Anxa1+ (Cohen's d = 2.05) than for Calb1+
> stimulation (d = 0.76).

In [ ]:
rows = []
for w in windows:
    for q in lrrk2.PAPER_POWERS:
        da = lrrk2.cohens_d(*[evoked[w].query("line == 'Anxa1' and power == @q and group == @g").value.dropna() for g in ("WT", "GS")])
        dc = lrrk2.cohens_d(*[evoked[w].query("line == 'Calb1' and power == @q and group == @g").value.dropna() for g in ("WT", "GS")])
        rows.append({"window": w, "power": q, "d_Anxa1": da, "d_Calb1": dc, "ratio": da / dc})
cd = pd.DataFrame(rows)
print("Cohen's d (WT minus G2019S over the pooled SD); paper at 0.5 mW: Anxa1 2.05, Calb1 0.76")
print(cd.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
m = cd.query("window == 'methods' and power == 0.5").iloc[0]
print(f"\nMethods window, 0.5 mW: Anxa1 d = {m.d_Anxa1:.3f} (tolerance 1.845 to 2.255), Calb1 d = {m.d_Calb1:.3f} (0.684 to 0.836), ratio {m.ratio:.2f}")

**Result: not reproduced.** At 0.5 mW, Cohen's d is 2.21 for Anxa1, within tolerance of 2.05, and 1.40 for Calb1,
against 0.76 (tolerance 0.684 to 0.836); their ratio is 1.58, below the factor of 2 the criterion asks. With the code's
window d is 2.06 and 1.25. The published integrals give 2.06 and 0.765 (see `source-data-link`), so the paper's values
follow from its own data, and the Calb1 difference is consistent with the Calb1 dF/F in the dandiset not being the one
the paper analysed. The Anxa1 effect is larger than the Calb1 effect at every power under the Methods, code and fixed
windows (ratio 1.29 to 1.74).

### Claims on data in other archives

These eight claims rest on data that the dandiset does not hold and that the paper deposited elsewhere. They are
`data: elsewhere` claims and are counted apart from the dandiset's. Each section names the archive.

### fig1d-lrrk2-enriched-vulnerable

> LRRK2 immunolabeling intensity is higher in Aldh1a1+ (vulnerable) than in Aldh1a1- SNc dopamine neurons (p = 0.0008).

Data: the per-section values on Zenodo (doi:10.5281/zenodo.19930567); the images are in BioImage Archive S-BIAD2070
(matched in the data check). The Imaris segmentation that produced the values is not re-run.

In [ ]:
t, p = archives.ttest(f1.pos, f1.neg)
print(f"Sections: {len(f1)} from {f1.mouse.nunique()} mice. Mean LRRK2 intensity, Aldh1a1+ {f1.pos.mean():.4f} (SEM {f1.pos.sem():.4f}),",
      f"Aldh1a1- {f1.neg.mean():.4f} (SEM {f1.neg.sem():.4f}); range of both {min(f1.pos.min(), f1.neg.min()):.3f} to {max(f1.pos.max(), f1.neg.max()):.3f}")
print(f"Two-tailed unpaired Student t-test: t = {t:.3f}, p = {p:.3g} (paper 0.0008); Cohen's d = {archives.cohens_d(f1.pos, f1.neg):.2f}")
print(f"Variants: Welch p = {archives.ttest(f1.pos, f1.neg, equal_var=False)[1]:.3g}; paired over sections p = {stats.ttest_rel(f1.pos, f1.neg).pvalue:.3g};",
      f"sections with Aldh1a1+ above Aldh1a1-: {int((f1.pos > f1.neg).sum())} of {len(f1)}")
mm = f1.groupby("mouse")[["neg", "pos"]].mean()
print("Per mouse (mean of sections):")
print(f1.groupby("mouse").agg(sections=("pos", "size"), pos=("pos", "mean"), neg=("neg", "mean"),
                              pos_above_neg=("pos", lambda s: int((s > f1.loc[s.index, "neg"]).sum()))).round(4).to_string())
print(f"Mouse means (n = 3 per type): unpaired p = {archives.ttest(mm.pos, mm.neg)[1]:.3g}, paired p = {stats.ttest_rel(mm.pos, mm.neg).pvalue:.3g}")

**Result: reproduced.** Over the 16 sections of the source data, LRRK2 intensity is higher in Aldh1a1+ than in
Aldh1a1- neurons (1.049 against 0.973), two-tailed unpaired Student p = 4.65e-06, Cohen's d 1.97; it is higher in 15
of 16 sections and in each of the three mice. The paper's p = 0.0008 is not that of the unpaired test over sections its
legend names; it matches a paired test over sections (p = 0.000759) or an unpaired test of the three mouse means
(p = 0.000822). The test starts from the paper's per-section values; the images were not segmented again.

### fig2b-rab3-phosphorylation

> LRRK2 kinase inhibition with MLi-2 changes 289 phosphopeptides in striatal synaptosomes of G2019S mice, including RAB3 T86.

Data: the peptide groups table in PRIDE PXD065006 (Proteome Discoverer abundances). The raw files (7.9 GB) need
Proteome Discoverer with Byonic, which are commercial, so the search is not re-run.

In [ ]:
rows, tests = [], {}
for reading, k in (("a", 3), ("b", 2)):
    for welch in (False, True):
        r = archives.volcano_test(pp, k) if not welch else None
        if welch:
            veh = [c for c in ab if "Vehicle" in c]
            mli = [c for c in ab if "MLi2" in c]
            a_, b_ = np.log2(pp[veh].to_numpy(float)), np.log2(pp[mli].to_numpy(float))
            keep = (np.isfinite(a_).sum(1) >= k) & (np.isfinite(b_).sum(1) >= k)
            r = pp.loc[keep, ["Annotated Sequence", "Modifications"]].copy()
            r["diff"] = np.nanmean(a_[keep], 1) - np.nanmean(b_[keep], 1)
            r["p"] = np.asarray(stats.ttest_ind(a_[keep], b_[keep], axis=1, equal_var=False, nan_policy="omit").pvalue, float)
            r["altered"] = (r.p <= 0.05) & (r["diff"].abs() > 0.58)
        else:
            tests[reading] = r
        rb = r[r["Annotated Sequence"].str.contains("YRTITTAYYR", na=False) & r.Modifications.str.contains(r"Phospho \[T3", na=False)]
        rows.append({"reading": reading, "test": "Welch" if welch else "Student", "peptides_tested": len(r),
                     "altered": int(r.altered.sum()), "higher_with_vehicle": int((r.altered & (r["diff"] > 0)).sum()),
                     "lower_with_vehicle": int((r.altered & (r["diff"] < 0)).sum()),
                     "rab3_T86_diff": float(rb["diff"].iloc[0]) if len(rb) else np.nan,
                     "rab3_T86_p": float(rb.p.iloc[0]) if len(rb) else np.nan,
                     "rab3_T86_altered": bool(rb.altered.iloc[0]) if len(rb) else False})
res2b = pd.DataFrame(rows)
print("Phosphopeptides altered at p <= 0.05 and |difference| > 0.58 (paper: 289, tolerance 260.1 to 317.9):")
print(res2b.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
for reading in ("a", "b"):
    r = tests[reading]
    print(f"Reading ({reading}): altered at p <= 0.05 alone {int((r.p <= 0.05).sum())}; with |difference| > 0.58 alone {int((r['diff'].abs() > 0.58).sum())}")

vol = s1["Volcano plot-phospho peptides"]
vol = vol.rename(columns={vol.columns[-2]: "Difference", vol.columns[-1]: "mlog10p"})
pub_alt = (vol.mlog10p >= -np.log10(0.05)) & (vol.Difference.abs() > 0.58)
print(f"\nThe paper's volcano table (Supplementary data 1): {len(vol)} peptides, {int(pub_alt.sum())} at the same thresholds",
      f"({int((pub_alt & (vol.Difference > 0)).sum())} higher with vehicle)")
pv = vol[vol["Annotated Sequence"].astype(str).str.contains("YRTITTAYYR")]
print("   its RAB3 T86 row:", pv[["Annotated Sequence", "Modifications", "Difference", "mlog10p"]].to_dict("records"))
# Variant: log2 abundances centred on each sample's median (the samples labelled MLi-2 have higher abundances overall).
L = np.log2(pp[ab].to_numpy(float))
print("\nMedian log2 abundance per sample:", dict(zip([c.split(": ")[-1] for c in ab], np.nanmedian(L, axis=0).round(2))))
norm = pp.copy()
norm[ab] = 2 ** (L - np.nanmedian(L, axis=0))
for reading, k in (("a", 3), ("b", 2)):
    r = archives.volcano_test(norm, k)
    rb = r[r["Annotated Sequence"].str.contains("YRTITTAYYR", na=False)]
    print(f"Variant, median-centred, reading ({reading}): {int(r.altered.sum())} altered ({int((r.altered & (r['diff'] > 0)).sum())} higher with",
          f"vehicle); RAB3 T86 difference (vehicle - MLi-2) {rb['diff'].iloc[0]:.3f}, p = {rb.p.iloc[0]:.3g}, altered {bool(rb.altered.iloc[0])}")
    if reading == "b":
        tests["b_norm"] = r
known = pp["Master Protein Descriptions"].astype(str).str.contains("Rab-10 |Rab-12 |Rab-8A |Rab-35 |Leucine-rich repeat serine", regex=True)
kn = archives.volcano_test(norm[known], 3)
print("Other LRRK2 targets in the table (median-centred, reading a):",
      [(m[:28], round(d_, 2), round(p_, 3)) for m, d_, p_ in zip(kn["Modifications in Master Proteins"], kn["diff"], kn.p)])

mg = vol.merge(tests["b"], on=["Annotated Sequence", "Modifications"], how="inner")
print(f"   matched to reading (b) by sequence and modifications: {len(mg)} of {len(vol)}; difference, ours against the table's:",
      f"r = {np.corrcoef(mg['diff'], mg.Difference)[0, 1]:.4f}, largest absolute gap {np.abs(mg['diff'] - mg.Difference).max():.3g};",
      f"-log10 p: r = {np.corrcoef(-np.log10(mg.p), mg.mlog10p)[0, 1]:.4f}, largest gap {np.abs(-np.log10(mg.p) - mg.mlog10p).max():.3g}")
mn = vol.merge(tests["b_norm"], on=["Annotated Sequence", "Modifications"], how="inner")
slope, icpt = np.polyfit(mn["diff"], mn.Difference, 1)
print(f"   against the median-centred reading (b): {len(mn)} matched; table difference = {slope:.3f} x (vehicle - MLi-2) + {icpt:.3f},",
      f"r = {np.corrcoef(mn['diff'], mn.Difference)[0, 1]:.4f}; -log10 p r = {np.corrcoef(-np.log10(mn.p), mn.mlog10p)[0, 1]:.4f};",
      f"peptides whose table difference has the sign of (vehicle - MLi-2): {int((np.sign(mn['diff']) == np.sign(mn.Difference)).sum())} of {len(mn)}")
# The table's difference and p-value recomputed exactly, under both assignments of the labels, on the peptides with all
# six values. A per-sample normalisation adds one constant to every difference, so the difference is compared up to
# a common offset; the p-value does not depend on the direction.
iv = [ab.index(c) for c in ab if "Vehicle" in c]
im = [ab.index(c) for c in ab if "MLi2" in c]
full = np.isfinite(L).all(axis=1)
for name, M in (("as deposited", L), ("median-centred", L - np.nanmedian(L, axis=0))):
    x = pp[["Annotated Sequence", "Modifications"]].copy()
    x["full"] = full
    x["veh_minus_mli"] = np.nanmean(M[:, iv], axis=1) - np.nanmean(M[:, im], axis=1)
    x["mlog10p_student"] = -np.log10(np.asarray(stats.ttest_ind(M[:, iv], M[:, im], axis=1).pvalue, float))
    x["mlog10p_welch"] = -np.log10(np.asarray(stats.ttest_ind(M[:, iv], M[:, im], axis=1, equal_var=False).pvalue, float))
    m_ = vol.merge(x, on=["Annotated Sequence", "Modifications"], how="inner")
    m_ = m_[m_.full]
    for label, sign in (("as labelled, vehicle - MLi-2", 1), ("labels swapped, MLi-2 - vehicle", -1)):
        gap = m_.Difference - sign * m_.veh_minus_mli
        off = gap.median()
        print(f"{name}, {label}: {len(m_)} peptides with all six values; table difference - ours = {off:.4f} + residual,",
              f"largest |residual| {(gap - off).abs().max():.2g}")
    for t in ("student", "welch"):
        g_ = (m_.mlog10p - m_[f"mlog10p_{t}"]).abs()
        print(f"   -log10 p, {t} test: median gap {g_.median():.3g}, largest {g_.max():.3g}, within 1e-3 for {int((g_ < 1e-3).sum())}")


**Result: not reproduced.** On the log2 abundances as deposited, 1694 (reading a) or 1810 (reading b)
phosphopeptides pass p <= 0.05 and |difference| > 0.58, against the paper's 289, and nearly all are lower with
vehicle (1673 and 1785), as the samples labelled MLi-2 have higher abundances overall (median log2 17.84 to 18.12,
against 17.25 to 17.47). The RAB3 T86 peptide passes (p = 0.00892) but is lower with vehicle (difference -1.31), so
neither part of the criterion holds. Centring each sample on its median, a step the criterion did not fix, gives 243
(a) and 282 (b) peptides, within tolerance for reading b, and RAB3 T86 is still higher with MLi-2 (difference -0.693,
p = 0.0238). The paper's own volcano table holds MLi-2 minus vehicle as the samples are labelled: on the 4340 peptides
with all six values its difference equals that difference plus a constant (-0.0043 after median-centring, -0.624 as
deposited) to within 1.6e-06, while against vehicle minus MLi-2 the residual reaches 13. Its p-values are close to
those of a t-test on the median-centred values but not equal (median gap in -log10 p 0.0088 for Welch, 0.0161 for
Student), so its test or normalisation differs in a way that does not affect the difference. The table holds 254
peptides at the legend's thresholds and gives RAB3 T86 +0.688, which the legend reads as higher with vehicle. As the samples are labelled in PRIDE and in
Supplementary data 1, the RAB3 T86 phosphopeptide is higher after MLi-2, the opposite of the claim, so either the
labels or the direction of the figure is swapped. The canonical LRRK2 site RAB10 T73 does not change (0.15,
p = 0.775), so the data cannot tell which.

### fig2eg-prab3-increased

> Phosphorylation of RAB3A and RAB3C is increased in the SNc of G2019S mice and returns to WT levels with MLi-2.

Data: the p-RAB3A/total ratios on Zenodo (doi:10.5281/zenodo.19930567). The uncropped blots are in the record's
`Westerbn blot.pdf`; the Methods do not say how bands were quantified, so they are not re-quantified.

In [ ]:
paper2 = {"RAB3A": {"GS vs WT": ("0.0202", "8.593"), "GS vs GS+MLi2": ("0.0069", "11.33")},
          "RAB3C": {"GS vs WT": ("0.0036", "14.76"), "GS vs GS+MLi2": ("0.0415", "7.398")}}
rows = []
for prot, d in f2.items():
    print(prot, {g: f"n {len(v)}, mean {v.mean():.3f} (SEM {v.std(ddof=1) / np.sqrt(len(v)):.3f})" for g, v in d.items()})
    print(f"   one-way ANOVA over the three groups: F = {stats.f_oneway(*d.values()).statistic:.3f}, p = {stats.f_oneway(*d.values()).pvalue:.3g}")
    for m in (3, 2):
        pairs = [("GS", "WT"), ("GS", "GS+MLi2"), ("GS+MLi2", "WT")][:m]
        c = archives.pooled_contrasts(d, pairs, m)
        for _, r in c.iterrows():
            rows.append({"protein": prot, "sidak_over": m, "pair": r.pair, "diff": r["diff"], "F(1,df)": r.F, "df": r.df,
                         "p": r.p, "p_sidak": r.p_sidak, "paper_p_sidak": paper2[prot].get(r.pair, ("", ""))[0],
                         "paper_F": paper2[prot].get(r.pair, ("", ""))[1],
                         "student_p": archives.ttest(d[r.pair.split(" vs ")[0]], d[r.pair.split(" vs ")[1]])[1]})
res2 = pd.DataFrame(rows)
print(res2.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
main2 = res2[(res2.sidak_over == 3) & res2.pair.isin(["GS vs WT", "GS vs GS+MLi2"])]
print("\nComparisons that hold (GS higher, Sidak over three pairs, p < 0.05):", int(((main2["diff"] > 0) & (main2.p_sidak < 0.05)).sum()), "of 4")

**Result: reproduced.** In the source data, p-RAB3A and p-RAB3C are higher in GS than in WT and than in GS with
MLi-2. With contrasts on the pooled variance, Sidak-adjusted over three pairs, the four comparisons give p = 0.0202
and 0.00688 (RAB3A) and 0.00358 and 0.0415 (RAB3C), with F = 8.593, 11.33, 14.76 and 7.398: the legend's values. GS with
MLi-2 does not differ from WT (adjusted p = 0.963 and 0.622), consistent with a return to WT levels. Adjusting over
two pairs gives smaller p-values (0.0135, 0.00459, 0.00239, 0.0279), so the legend's numbers follow three pairs. The
blots were not quantified again.

### fig3d-rim-binds-unphosphorylated

> RIM1 and RIM2 bind unphosphorylated RAB3A in preference to RAB3A phosphorylated at T86, while RILPL1 and RILPL2 prefer the
> phosphorylated form.

Data: the DIA-NN protein intensities of PRIDE PXD065125, six pulldowns with each bait, paired in run order (data check).

In [ ]:
phos_cols = [f"phos_{k}" for k in range(1, 7)]
unphos_cols = [f"unphos_{k}" for k in range(1, 7)]
rows = []
for gene in ("Rims1", "Rims2", "Rilpl1", "Rilpl2"):
    for _, r in pull[pull.Genes == gene].iterrows():
        rows.append({"gene": gene, "protein_group": r["Protein.Group"], "n_unphos": int(r[unphos_cols].notna().sum()),
                     "n_phos": int(r[phos_cols].notna().sum()), **archives.paired_test(r)})
res3 = pd.DataFrame(rows)
res3["values"] = res3.n_unphos + res3.n_phos
res3["used"] = False
for gene, g in res3.groupby("gene", sort=False):
    res3.loc[g["values"].idxmax(), "used"] = True
print("Paired two-sided t-test of log2 intensity, RAB3A against p-RAB3A pulldowns (paper: RIM2 p = 0.011, RILPL1 p = 0.002):")
print(res3.to_string(index=False, float_format=lambda v: f"{v:.3g}"))
print("Groups shared between genes:", pull[pull.Genes.str.contains("Rims1|Rims2|Rilpl", na=False) & pull.Genes.str.contains(";")][["Genes", "Protein.Group"]].to_dict("records"))
used = res3[res3.used].set_index("gene")
holds = {g: bool((used.loc[g, "log2FC_unphos_minus_phos"] > 0) == g.startswith("Rims") and used.loc[g, "p"] < 0.05) for g in used.index}
print("Holds under the criterion's paired test (RIMs higher with RAB3A, RILPLs higher with p-RAB3A, p < 0.05):", holds, "->",
      sum(holds.values()), "of 4")

# Variant: the same paired test on intensities without the log.
lin = {g: stats.ttest_rel(2 ** pull[pull.Genes == g][unphos_cols].iloc[0].to_numpy(float),
                          2 ** pull[pull.Genes == g][phos_cols].iloc[0].to_numpy(float)).pvalue for g in ("Rims2", "Rilpl1")}
print("Variant, paired t-test on linear intensities (the first group of each gene):", {k: round(float(v), 4) for k, v in lin.items()})
r2 = pull[pull.Genes == "Rilpl2"].iloc[0]
print("RILPL2 values with RAB3A:", int(r2[unphos_cols].notna().sum()), "of 6; with p-RAB3A:", int(r2[phos_cols].notna().sum()), "of 6,",
      "log2 from", round(float(r2[phos_cols].min()), 2), "to", round(float(r2[phos_cols].max()), 2),
      "; the paired test has no pair, so the criterion's test does not find RILPL2 higher with p-RAB3A")

# Variant: the Methods' imputation, then an unpaired Student t-test, on every protein group.
rng3 = np.random.default_rng(1933)
imp = archives.impute(pull, phos_cols + unphos_cols, rng3)
t3 = stats.ttest_ind(imp[unphos_cols], imp[phos_cols], axis=1)
imp["log2FC"] = imp[unphos_cols].mean(axis=1) - imp[phos_cols].mean(axis=1)
imp["p"] = t3.pvalue
print("\nVariant, imputation (seeded) and unpaired t-test:")
print(imp[imp.Genes.isin(["Rims1", "Rims2", "Rilpl1", "Rilpl2"])][["Genes", "Protein.Group", "log2FC", "p"]]
      .to_string(index=False, float_format=lambda v: f"{v:.3g}"))
lim = pd.read_csv(io.BytesIO(archives.fetch(archives.PULLDOWN_LIMMA_URL)[0]))
print(f"\nThe deposited limma table ({len(lim)} protein groups, comparison {lim.Comparison.unique().tolist()}):")
print(lim[lim.Genes.isin(["Rims1", "Rims2", "Rilpl1", "Rilpl2"])][["Genes", "Protein.Group", "logFC", "P.Value", "adj.P.Val"]].to_string(index=False))
print("   groups at P.Value < 0.05 with logFC >= 1 (more with RAB3A):", int(((lim["P.Value"] < 0.05) & (lim.logFC >= 1)).sum()),
      "; with logFC <= -1 (more with p-RAB3A):", int(((lim["P.Value"] < 0.05) & (lim.logFC <= -1)).sum()), "(paper: 3480 proteins, 69 and 16)")
print("   the same with adj.P.Val < 0.05:", int(((lim["adj.P.Val"] < 0.05) & (lim.logFC >= 1)).sum()), "and",
      int(((lim["adj.P.Val"] < 0.05) & (lim.logFC <= -1)).sum()))
rilpl2 = imp[imp.Genes == "Rilpl2"].iloc[0]
holds_imp = dict(holds, Rilpl2=bool(rilpl2.log2FC < 0 and rilpl2.p < 0.05))
print("Variant, RILPL2 from the imputation:", holds_imp, "->", sum(holds_imp.values()), "of 4")

**Result: partially reproduced.** In the PRIDE pulldown matrix, RIM1 (Q99NE5-7) and RIM2 are higher with RAB3A (log2
differences 2.42 and 2.17, paired p = 0.00361 and 0.0124) and RILPL1 is higher with p-RAB3A (-6.02, p = 9.8e-05).
RILPL2 is detected in all six p-RAB3A pulldowns and in none of the RAB3A ones, so the criterion's paired test has no
pair and does not find it higher with p-RAB3A: three of four comparisons hold. The data point the way of the claim for
RILPL2: with the Methods' imputation, a variant, it is higher with p-RAB3A (log2 difference -7.92, p = 1.21e-08), and
the deposited limma table gives logFC -7.58. The other two RIM1 groups agree (paired p = 0.0196 and 0.00984). The
paper's p-values for RIM2 and RILPL1 (0.011 and 0.002) are those of the paired test on intensities without the log
(0.0109 and 0.0022). The deposited limma table agrees in direction for every group. At the text's p-value < 0.05 and
|log2FC| >= 1 it gives 121 and 32 proteins; the paper's 69 and 16 are the counts at adjusted p < 0.05.

### fig4d-fewer-active-zones

> Fewer TH+/VAMP2+ striatal synaptosomes from G2019S mice contain bassoon, and the remaining bassoon clusters are larger.

Data: (A) the per-area values on Zenodo; (B) the images of BioImage Archive S-BIAD2074, measured in the data check with
the Methods' FIJI procedure as re-implemented in `archives.py`.

In [ ]:
rows = []
for panel, label in (("4D", "percent of TH+/VAMP2+ with bassoon"), ("4E", "bassoon area (um2)")):
    d = f4de[f4de.panel == panel]
    gs, wt = d[d.group == "GS"].value, d[d.group == "WT"].value
    reps = d.groupby(["group", "replicate"]).value.mean()
    rows.append({"part": "A", "panel": panel, "measure": label, "n_WT": len(wt), "n_GS": len(gs), "mean_WT": wt.mean(),
                 "mean_GS": gs.mean(), "p": archives.ttest(gs, wt)[1], "d_GS_minus_WT": archives.cohens_d(gs, wt),
                 "p_welch": archives.ttest(gs, wt, equal_var=False)[1], "p_mouse_means": archives.ttest(reps["GS"], reps["WT"])[1]})
d = f4de[(f4de.panel == "4D") & ~((f4de.group == "GS") & (f4de.value < 1))]
print("(A) Source data, two-tailed unpaired Student t-tests over areas (paper: 4D p < 0.0001, 4E p = 0.0301):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print(f"   4D without the GS value under 1 %: p = {archives.ttest(d[d.group == 'GS'].value, d[d.group == 'WT'].value)[1]:.3g}")
print("   4D per replicate (mean):", f4de[f4de.panel == "4D"].groupby(["replicate", "group"]).value.mean().round(2).unstack().to_dict("index"))
print("   4E per replicate (mean):", f4de[f4de.panel == "4E"].groupby(["replicate", "group"]).value.mean().round(4).unstack().to_dict("index"))

rows = []
for folder, col, label in (("BSN", "percent", "percent with bassoon"), ("BSN-area", "area_overlap", "area, reading (i) overlap"),
                           ("BSN-area", "area_bassoon", "area, reading (ii) bassoon particle")):
    d = syn[(syn.folder == folder)].dropna(subset=[col])
    gs, wt = d[d.group == "GS"][col], d[d.group == "WT"][col]
    mice = d.groupby(["group", "mouse"])[col].mean()
    rows.append({"part": "B", "folder": folder, "measure": label, "images_WT": len(wt), "images_GS": len(gs),
                 "dropped": int((syn.folder == folder).sum() - len(d)), "mean_WT": wt.mean(), "mean_GS": gs.mean(),
                 "min": d[col].min(), "max": d[col].max(), "p": archives.ttest(gs, wt)[1], "d_GS_minus_WT": archives.cohens_d(gs, wt),
                 "p_mouse_means": archives.ttest(mice["GS"], mice["WT"])[1]})
print("\n(B) Images, the same tests over images (dropped: images with no particle to measure):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
print("   per mouse (mean over images):")
print(syn.groupby(["folder", "group", "mouse"])[["percent", "area_overlap", "area_bassoon", "th_vamp2", "vamp2_pos"]].mean().round(3).to_string())

**Result: partially reproduced.** (A) In the source data, fewer TH+/VAMP2+ synaptosomes contain bassoon in GS
(12.0 against 21.4 %, p = 6.38e-05; 0.000118 without the value under 1 %) and the bassoon area is larger (0.0862
against 0.0750 um2, p = 0.0301, the paper's value), so part A holds. Both tests treat imaging areas as independent;
with one value per mouse (3 or 4 per group) p = 0.263 and 0.474. (B) The Methods' FIJI procedure as re-implemented here
finds neither effect in the images: 48.8 against 52.4 % with bassoon (GS against WT, p = 0.459), and a bassoon area
of 0.0682 against 0.0696 um2 (overlap, p = 0.805) or 0.132 against 0.141 um2 (bassoon particle, p = 0.206). The
re-implementation keeps far fewer synaptosomes than the paper (median 183 VAMP2 particles per image against about
1000; the threshold alone gives a median of 1186 objects, and the size limit removes most of them) and finds
percentages more than twice the paper's. Part B may therefore fail because of this attempt's segmentation, and its
null result is weak evidence against the claim.

### fig4i-rim1-reduced

> RIM1 is reduced in striatal synaptosomes of G2019S mice (p = 0.0245) but not in other striatal fractions or the SNc.

Data: the band values on Zenodo (doi:10.5281/zenodo.19930567); the blots are not re-quantified.

In [ ]:
names = {"4G": "synaptophysin/actin, synaptosomes", "4H": "Figure 4H protein, synaptosomes", "4I": "RIM1, synaptosomes",
         "4J": "RIM1, S1 fraction", "4L": "RIM1, SNc"}
rows = []
for k, v in blots.items():
    t, p = archives.ttest(v["GS"], v["WT"])
    rows.append({"panel": k, "what": names[k], "n_WT": len(v["WT"]), "n_GS": len(v["GS"]), "mean_WT": v["WT"].mean(),
                 "mean_GS": v["GS"].mean(), "t": t, "p": p, "p_welch": archives.ttest(v["GS"], v["WT"], equal_var=False)[1],
                 "d_GS_minus_WT": archives.cohens_d(v["GS"], v["WT"]),
                 "range": f"{min(v['WT'].min(), v['GS'].min()):.3f} to {max(v['WT'].max(), v['GS'].max()):.3f}"})
print("Two-tailed unpaired Student t-tests, GS against WT (paper: 4I p = 0.0245; 4G, 4J and 4L not significant):")
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

**Result: reproduced.** RIM1 in synaptosomes is lower in GS (0.206 against 0.319, two-tailed p = 0.0245, the paper's
value; Cohen's d -1.53), and RIM1 in the S1 fraction (p = 0.896) and in the SNc (p = 0.936) does not differ.
Synaptophysin (4G, p = 0.445) and the protein of 4H (p = 0.634) do not differ either. The test starts from the
deposited band values.

### fig5-bassoon-vulnerable-axons

> Bassoon cluster density is reduced and cluster volume increased in Aldh1a1+ but not Aldh1a1- dopamine axons of G2019S mice,
> and the same holds in Anxa1+ axons.

Data: the per-section values on Zenodo; the 542 GB of SIM images of S-BIAD2071 are matched by file name in the data check.

In [ ]:
dup_cols = set(wide[wide.round(9).duplicated(keep="first")].index.get_level_values("column"))
print("Sections counted once (columns of the source sheet dropped as repeats):", sorted(dup_cols))
paper5 = {"5D": {"cell": ("12.75", "0.0006"), "group": ("1.277", "0.2622"), "interaction": ("4.838", "0.0311"), "sidak+": "0.0344"},
          "5E": {"cell": ("4.607", "0.0354"), "group": ("1.277", "0.0252"), "interaction": ("0.8688", "0.3545"), "sidak+": "0.0385"}}
parts = {}
for variant, data in (("main", f5[~f5.column.isin(dup_cols)]), ("as deposited", f5)):
    for panel in ("5D", "5E"):
        d = data[data.panel == panel].dropna(subset=["value"])
        anova, mse, dfr = archives.twoway_anova(d)
        sid = archives.sidak_within(d, mse, dfr)
        print(f"\n{panel}, {variant}: two-way ANOVA (paper: cell type F {paper5[panel]['cell'][0]}, p {paper5[panel]['cell'][1]};",
              f"genotype F {paper5[panel]['group'][0]}, p {paper5[panel]['group'][1]}; interaction F {paper5[panel]['interaction'][0]},",
              f"p {paper5[panel]['interaction'][1]}; Sidak Aldh1a1+ p {paper5[panel]['sidak+']})")
        print(anova.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        print(sid.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
        if variant == "main":
            s = sid.set_index("cell")
            sign = -1 if panel == "5D" else 1
            parts[f"{panel} Aldh1a1+"] = bool(sign * (s.loc["Aldh1a1+", "mean_GS"] - s.loc["Aldh1a1+", "mean_WT"]) > 0 and s.loc["Aldh1a1+", "p_sidak"] < 0.05)
            parts[f"{panel} Aldh1a1- not different"] = bool(s.loc["Aldh1a1-", "p_sidak"] >= 0.05)
z = f5[(f5.panel == "5D") & (f5.value == 0)]
print("\nZero values in 5D:", z[["group", "mouse", "section", "cell"]].to_dict("records"))
print("\nAnxa1 (5G density, 5H volume), two-tailed unpaired Student t-tests (paper: 5G p = 0.0374, 5H p = 0.0148):")
for panel, sign in (("5G", -1), ("5H", 1)):
    d = f5gh[f5gh.panel == panel]
    gs, wt = d[d.group == "GS"].value, d[d.group == "WT"].value
    t, p = archives.ttest(gs, wt)
    print(f"   {panel}: n WT {len(wt)}, GS {len(gs)}; mean WT {wt.mean():.5g}, GS {gs.mean():.5g}; t = {t:.3f}, p = {p:.3g};",
          f"Welch p = {archives.ttest(gs, wt, equal_var=False)[1]:.3g}; d (GS - WT) = {archives.cohens_d(gs, wt):.2f}")
    parts[f"{panel} Anxa1"] = bool(sign * (gs.mean() - wt.mean()) > 0 and p < 0.05)
print("\nParts that hold:", parts, "->", sum(parts.values()), "of 6")

**Result: partially reproduced.** Five of the six parts hold. With section 5 of WT mouse 1723 counted once (its
four values equal those of section 4), Aldh1a1+ density is lower in GS (0.164 against 0.217) but the Sidak-adjusted p
is 0.058, so part 1 fails; Aldh1a1- density does not differ (p = 0.713). Bassoon volume is larger in GS Aldh1a1+ axons
(Sidak p = 0.0425) and not in Aldh1a1- axons (p = 0.699). In Anxa1+ axons density is lower (p = 0.0374) and volume
larger (p = 0.0148), the paper's values. With the section counted twice, as deposited, all six parts hold and the
ANOVA gives the legend's numbers (5D: F 12.75, 1.277, 4.838, Sidak p 0.0344; 5E: Sidak p 0.0385). The legend's
genotype F for 5E (1.277) repeats that of 5D; the deposited values give F = 5.234 with the legend's p = 0.0252. The
analysis starts from the per-section values; the SIM images were not analysed again.

### fig5j-no-axon-loss

> The volume occupied by TH+ fibres in the dorsal striatum does not differ between G2019S and WT mice.

Data: the per-section values on Zenodo; the images of S-BIAD2072 are matched by file name in the data check.

In [ ]:
gs, wt = f5j[f5j.group == "GS"].value, f5j[f5j.group == "WT"].value
t, p = archives.ttest(gs, wt)
print(f"Sections: WT {len(wt)}, GS {len(gs)}; mean TH volume (normalised) WT {wt.mean():.4f} (SEM {wt.sem():.4f}), GS {gs.mean():.4f} (SEM {gs.sem():.4f});",
      f"range {f5j.value.min():.3f} to {f5j.value.max():.3f}")
print(f"Two-tailed unpaired Student t-test: t = {t:.3f}, p = {p:.3g}; difference of means (GS - WT) {gs.mean() - wt.mean():.4f};",
      f"Cohen's d (GS - WT) {archives.cohens_d(gs, wt):.2f}; Welch p = {archives.ttest(gs, wt, equal_var=False)[1]:.3g}")
print("Per animal-column group (the sheet gives one animal column for both genotypes):")
print(f5j.groupby(["animal_column", "group"]).value.agg(["size", "mean"]).round(4).unstack().to_string())

**Result: reproduced.** The paper's test does not find a difference in TH volume: two-tailed p = 0.117 over 12
sections per genotype. GS is 7.1 % lower (0.929 against 1.000; Cohen's d -0.67).

## Summary

| Claim | Data | Outcome | Attempt -01 |
|---|---|---|---|
| `source-data-link` | dandiset with inputs | not reproduced (Anxa1 links; Calb1 does not) | not reproduced |
| `fig6e-anxa1-evoked-reduced` | dandiset | reproduced | reproduced |
| `fig6e-anxa1-evoked-per-mouse` | dandiset | reproduced | reproduced |
| `supp7c-anxa1-evoked-all-powers` | dandiset | reproduced | reproduced |
| `fig6f-anxa1-spontaneous-reduced` | dandiset | reproduced | reproduced |
| `fig6f-anxa1-spontaneous-per-mouse` | dandiset | partially reproduced | partially reproduced |
| `supp8c-calb1-spontaneous-not-reduced` | dandiset | reproduced (p = 0.0509, at the threshold; 0.040 to 0.079 as the window moves) | reproduced |
| `supp8e-calb1-evoked-reduced` | dandiset | reproduced | reproduced |
| `supp8e-calb1-few-mice-reduced` | dandiset | partially reproduced | partially reproduced |
| `cohens-d-anxa1-larger` | dandiset | not reproduced | not reproduced |
| `fig1d-lrrk2-enriched-vulnerable` | elsewhere | reproduced | blocked, data missing |
| `fig2b-rab3-phosphorylation` | elsewhere | not reproduced | blocked, data missing |
| `fig2eg-prab3-increased` | elsewhere | reproduced | blocked, data missing |
| `fig3d-rim-binds-unphosphorylated` | elsewhere | partially reproduced | blocked, data missing |
| `fig4d-fewer-active-zones` | elsewhere | partially reproduced | blocked, data missing |
| `fig4i-rim1-reduced` | elsewhere | reproduced | blocked, data missing |
| `fig5-bassoon-vulnerable-axons` | elsewhere | partially reproduced | blocked, data missing |
| `fig5j-no-axon-loss` | elsewhere | reproduced | blocked, data missing |

The photometry outcomes are those of attempt -01: the data, the code and the numbers are the same. After the first
verification round, `fig3d-rim-binds-unphosphorylated` was changed from reproduced to partially reproduced, as its
criterion's paired test cannot score RILPL2. The eight claims
of Figures 1 to 5 were blocked in attempt -01 under pilot-6; under pilot-7 they are tested on the data the paper
deposited elsewhere and are counted apart from the dandiset's claims.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences
from the paper's method.

How the analysis differs from the paper's:

- The evoked integral is taken over the light-on period, as the Methods state; the paper's numbers come from the code's
  window, 0.1 s before to 1.0 s after onset (its published integrals equal that measure times 0.0001). Both are shown.
- Spontaneous transients are measured by their integral, as the Methods state; the paper's published values are 0.005
  times the integral divided by the duration of each transient. Both are shown; the published measure gives the
  paper's Figure 6F p-value and per-mouse count exactly.
- Per-mouse tests are shown under four readings, as the Methods (Welch), the legends (one-tailed) and the code
  (Student, two-tailed) disagree.
- The window `chunking_index.m` computes as written starts 1 s after onset; the source data show the paper used the
  intended window starting 1 s before onset. The literal window is shown as a variant only.
- The claims of Figures 1, 2E to 2G, 4I and 5 start from the paper's deposited per-section, per-area and per-blot
  values, as the Imaris analyses and the blot quantification cannot be repeated here. Figure 2B starts from the
  deposited peptide abundances without normalisation, which the criterion did not fix; a median-centred variant is
  shown. Figure 3 uses log2 intensities, where the paper's p-values are on intensities without the log; RILPL2 has no
  pair for the paired test, so it does not hold under the criterion, and the Methods' imputation is shown as a variant. Figure 4D and 4E add a re-implementation
  of the Methods' FIJI procedure in Python. One Figure 5 section entered twice is counted once.

What was found about the dandiset:

- The Anxa1 recordings are those behind Figure 6 and Supplementary Figure 7: the matched published traces equal the
  dandiset's dF/F to within 5.7e-13, and the evoked integrals and transient counts match, except one stimulation of
  mouse 4398 that the processed series cuts off 0.8 s after onset.
- The Calb1 dF/F in the dandiset is not the trace behind Supplementary Figure 8. None of the 384 published traces or
  integrals match it, no epoch of any power matches a published trace, and recomputing from the raw signal with any
  single background fraction does not close the gap. The raw signal is there, but the processing the paper used for
  Calb1 is not, so the Calb1 results here are of the dandiset's processing, not the paper's.
- The epochs table describes one kind of train and four powers; the files hold two kinds of train and eight powers, and
  two Anxa1 GS mice received the paper's powers as 0.33 s trains of 9 ms pulses; in these two mice the response does
  not follow the labelled power.
- The dF/F unit is given as `a.u.` for a series in percent; two trains have a pulse length of 0; the four excluded mice
  are left out without a note; the archive summary lists a behavioral approach; and the paper does not cite the
  dandiset. The dandiset does not link to the Zenodo, PRIDE and BioImage Archive deposits that hold the rest of the
  paper's data. None of these findings of attempt -01 is fixed.

What was found about the data in other archives:

- In PRIDE PXD065006 and Supplementary data 1, the RAB3 T86 phosphopeptide is higher in the samples labelled MLi-2,
  and the paper's volcano table is MLi-2 minus vehicle (to within 1.6e-06 after a constant), while the legend reads a
  positive difference as higher with vehicle. The table gives 254 peptides at the legend's thresholds, not 289.
- The paper's counts of 69 and 16 RAB3A-preferring and p-RAB3A-preferring proteins are at adjusted p < 0.05, where the
  text says p-value < 0.05; its Figure 1D p-value is not that of the test the legend names.
- One Figure 5D/5E section appears twice in the source data; counting it once moves one comparison to p = 0.058.
  Figure 4E has 51 GS areas against the legend's 57, and the 5E legend repeats the 5D genotype F.
- The source data do not identify the GS animals of Figures 5G, 5H and 5J, and the image studies hold images and mice
  that the source data do not (an extra WT mouse in the Anxa1 study, mouse 1923 against 1723, more images than
  sections), without saying which images the figures used. The OME metadata of three synaptosome images of
  S-BIAD2074 give 488 nm for all three channels.

In [ ]:
print(f"total run time {time.time() - started:.0f} s")